# Sistema Agentivo para Análise Exploratória e Controle de Qualidade de Dados de Monitoramento da Qualidade da Água

## 1. Contexto do projeto

O objetivo deste projeto é construir um sistema de trabalho capaz de conduzir, de forma consistente, verificável e reutilizável, uma análise exploratória e um processo de controle de qualidade de dados de monitoramento da qualidade das águas.

A proposta não é substituir a análise técnica por uma resposta de modelo de linguagem. O sistema combina operações determinísticas realizadas por código com uma camada de interpretação, permitindo maior rastreabilidade, validação dos resultados e controle sobre as conclusões produzidas.

---

## 2. Domínio e escopo

**Domínio:** monitoramento da qualidade das águas superficiais em Minas Gerais.

**Usuário-alvo:** analista ambiental ou profissional responsável pela análise de dados de monitoramento.

**Pergunta orientadora:**

> Quais problemas de qualidade e completude dos dados, padrões temporais e observações que merecem investigação podem ser identificados automaticamente a partir de uma série histórica de monitoramento?

**Decisão apoiada:**

Identificar períodos, parâmetros e observações que devem receber atenção ou investigação adicional antes da utilização dos dados em análises ambientais.

O sistema está deliberadamente restrito a esse domínio e família de dados. Ele não pretende produzir diagnóstico ambiental definitivo ou substituir a avaliação de especialistas.

---

## 3. Dados utilizados

Foi utilizada a base histórica de dados de qualidade das águas do Programa Águas de Minas, do Instituto Mineiro de Gestão das Águas (IGAM).

O arquivo utilizado nesta demonstração é:

**Dados_Qualidade_das_Aguas_ate_2019.xlsx**

A análise utiliza principalmente a planilha **"SH ATÉ DEZ 2019"**.

Características observadas na base:

- 39.615 registros de amostragem;
- 769 estações;
- 96 parâmetros ambientais;
- período de 1997 a 2019;
- estrutura com colunas de valor e respectivas colunas de sinal;
- dados físico-químicos e microbiológicos;
- presença de valores ausentes em diferentes níveis de cobertura.

A base completa é preservada e os dados ausentes não são removidos automaticamente.

---

## 4. Recorte analítico

Embora a base contenha 96 parâmetros, a auditoria exploratória utiliza um conjunto reduzido de 11 parâmetros principais:

- pH in loco;
- Turbidez;
- Oxigênio dissolvido;
- Temperatura da água;
- Condutividade elétrica in loco;
- Demanda Bioquímica de Oxigênio;
- Fósforo total;
- Nitrato;
- Sólidos totais;
- Nitrogênio amoniacal total;
- Coliformes termotolerantes.

A seleção considera relevância para a caracterização da qualidade da água, cobertura na série histórica, diversidade de dimensões ambientais e viabilidade de demonstração do sistema.

Os demais parâmetros permanecem preservados na base e podem ser incorporados a análises futuras conforme a pergunta analítica e a disponibilidade dos dados.

---

## 5. Estratégia do sistema

O fluxo desenvolvido neste projeto é:

**Entrada dos dados**
→ **Auditoria e controle de qualidade**
→ **Padronização**
→ **Análise exploratória**
→ **Validação dos achados**
→ **Relatório**

As operações críticas, como contagem, cálculo de cobertura, identificação de duplicidades e estatísticas, são realizadas por código determinístico.

A camada de linguagem pode ser utilizada para interpretar os resultados e comunicar os achados, mas não substitui as verificações executadas pelo sistema.

---

## 6. Princípios de confiabilidade

O sistema foi desenvolvido com alguns princípios:

- preservar os dados originais;
- preservar valores ausentes;
- preservar os sinais associados às medições;
- não classificar automaticamente valores extremos como erros;
- diferenciar evidência de interpretação;
- não inferir causalidade a partir de associações ou lacunas;
- registrar as verificações realizadas;
- bloquear ou sinalizar etapas quando os critérios mínimos não forem atendidos.

---

## 7. Limitações e fora do escopo

Este projeto não pretende:

- estabelecer relações de causalidade;
- determinar a causa de dados ausentes;
- produzir diagnóstico ambiental definitivo;
- determinar risco à saúde;
- afirmar conformidade regulatória;
- substituir avaliação técnica especializada;
- realizar previsão futura sem uma pergunta e dados adequados para modelagem preditiva.

Os resultados devem ser interpretados dentro do escopo e das limitações da base analisada.

In [ ]:
# Carregar base de dados
from pathlib import Path
import pandas as pd

caminhos_possiveis = [
    Path("data") / "Dados_Qualidade_das_Aguas_ate_2019.xlsx",
    Path("../data") / "Dados_Qualidade_das_Aguas_ate_2019.xlsx"
]

arquivo = next(
    (caminho for caminho in caminhos_possiveis if caminho.exists()),
    None
)

if arquivo is None:
    raise FileNotFoundError(
        "Base de dados não encontrada. "
        "Verifique se o arquivo está na pasta data/ do projeto."
    )

xls = pd.ExcelFile(arquivo)

print(f"Arquivo utilizado: {arquivo}")
print(xls.sheet_names)

['SH ATÉ DEZ 2019', 'SCQA CIANO E ECOTOX 2018', 'SCQA ECOTOX E CIANO ANUAL 2019', 'SCQA IQA 2018', 'SCQA IQA TRIMESTRAL 2019', 'SCQA CT 2018', 'SCQA CT ANUAL 2019', 'SCQA IET 2018', 'SCQA IET ANUAL 2019']


In [ ]:
# Verificar abas do arquivo
for aba in xls.sheet_names:
    df = pd.read_excel(arquivo, sheet_name=aba)

    print(f"\nABA: {aba}")
    print(f"Linhas: {df.shape[0]}")
    print(f"Colunas: {df.shape[1]}")


ABA: SH ATÉ DEZ 2019
Linhas: 39615
Colunas: 195

ABA: SCQA CIANO E ECOTOX 2018
Linhas: 342
Colunas: 7

ABA: SCQA ECOTOX E CIANO ANUAL 2019
Linhas: 1328
Colunas: 7

ABA: SCQA IQA 2018
Linhas: 630
Colunas: 6

ABA: SCQA IQA TRIMESTRAL 2019
Linhas: 650
Colunas: 6

ABA: SCQA CT 2018
Linhas: 626
Colunas: 6

ABA: SCQA CT ANUAL 2019
Linhas: 649
Colunas: 6

ABA: SCQA IET 2018
Linhas: 630
Colunas: 6

ABA: SCQA IET ANUAL 2019
Linhas: 650
Colunas: 6


In [ ]:
# Verificar estrutura das unidades de observação
df = pd.read_excel(
    arquivo,
    sheet_name="SH ATÉ DEZ 2019"
)

df.head()

,Estação,Data de Amostragem,Hora de Amostragem,"Sinal 2,4,6 Triclorofenol","2,4,6 Triclorofenol",Sinal Alcalinidade de bicarbonato,Alcalinidade de bicarbonato,Sinal Alcalinidade total,Alcalinidade total,Sinal Aldrin + Dieldrin,...,Sinal Transparência,Transparência,Sinal Trifluoralina,Trifluoralina,Sinal Turbidez,Turbidez,Sinal Vanádio total,Vanádio total,Sinal Zinco total,Zinco total
0,AV005,2003-01-13,11:20:00,NaN,NaN,NaN,NaN,NaN,8.1,NaN,...,NaN,NaN,NaN,NaN,NaN,31.90,NaN,NaN,<,0.02
1,AV005,2003-04-01,10:35:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,5.30,NaN,NaN,NaN,NaN
2,AV005,2003-07-01,10:40:00,NaN,NaN,NaN,NaN,NaN,10.8,NaN,...,NaN,NaN,NaN,NaN,NaN,1.87,NaN,NaN,<,0.02
3,AV005,2003-10-02,11:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2.45,NaN,NaN,NaN,NaN
4,AV005,2004-01-12,11:00:00,NaN,NaN,NaN,NaN,NaN,10.6,NaN,...,NaN,NaN,NaN,NaN,NaN,9.14,NaN,NaN,<,0.02


In [ ]:
df[
    [
        "Estação",
        "Data de Amostragem",
        "Hora de Amostragem"
    ]
].head(10)

,Estação,Data de Amostragem,Hora de Amostragem
0,AV005,2003-01-13,11:20:00
1,AV005,2003-04-01,10:35:00
2,AV005,2003-07-01,10:40:00
3,AV005,2003-10-02,11:05:00
4,AV005,2004-01-12,11:00:00
5,AV005,2004-04-01,11:10:00
6,AV005,2004-07-05,10:55:00
7,AV005,2004-10-01,10:35:00
8,AV005,2006-10-02,10:50:00
9,AV005,2007-01-22,10:10:00


In [ ]:
# Cobertura temporal dos dados
df["Data de Amostragem"] = pd.to_datetime(
    df["Data de Amostragem"],
    errors="coerce"
)

print(df["Data de Amostragem"].min())
print(df["Data de Amostragem"].max())

1997-07-02 00:00:00
2019-12-12 00:00:00


In [ ]:
# Registros por ano
df["ano"] = df["Data de Amostragem"].dt.year

df["ano"].value_counts().sort_index()

,count
ano,
1997,421
1998,745
1999,795
2000,959
2001,962
2002,971
2003,1159
2004,1161
2005,1156


In [ ]:
# Quantidade de estações
df["Estação"].nunique()

769

In [ ]:
df["Estação"].value_counts().head(20)

,count
Estação,
BV141,174
BV146,174
BV139,174
BV149,174
BV152,174
BV105,174
BV148,174
BV156,173
BV063,163


In [ ]:
df.groupby("ano")["Estação"].nunique()

,Estação
ano,
1997,195
1998,201
1999,202
2000,242
2001,242
2002,248
2003,278
2004,278
2005,319


In [ ]:
# Verificar parâmetros
colunas = df.columns.tolist()

for coluna in colunas:
    print(coluna)

Estação
Data de Amostragem
Hora de Amostragem
Sinal 2,4,6 Triclorofenol
2,4,6 Triclorofenol
Sinal Alcalinidade de bicarbonato
Alcalinidade de bicarbonato
Sinal Alcalinidade total
Alcalinidade total
Sinal Aldrin + Dieldrin
Aldrin + Dieldrin
Sinal Alumínio dissolvido
Alumínio dissolvido
Sinal Alumínio total
Alumínio total
Sinal Arsênio Dissolvido
Arsênio Dissolvido
Sinal Arsênio total
Arsênio total
Sinal Atrazina
Atrazina
Sinal Bário total
Bário total
Sinal Boro dissolvido
Boro dissolvido
Sinal Boro total
Boro total
Sinal Cádmio total
Cádmio total
Sinal Cálcio total
Cálcio total
Sinal Chumbo total
Chumbo total
Sinal Cianeto Livre
Cianeto Livre
Sinal Cianeto total
Cianeto total
Sinal Clordano - cis + trans
Clordano - cis + trans
Sinal Cloreto total
Cloreto total
Sinal Clorofila a
Clorofila a
Sinal Cobre dissolvido
Cobre dissolvido
Sinal Cobre total
Cobre total
Sinal Coliformes termotolerantes
Coliformes termotolerantes
Sinal Coliformes totais
Coliformes totais
Sinal Condição de tempo
Cond

In [ ]:
colunas_sinal = [
    c for c in df.columns
    if c.startswith("Sinal")
]

print(colunas_sinal)

['Sinal 2,4,6 Triclorofenol', 'Sinal Alcalinidade de bicarbonato', 'Sinal Alcalinidade total', 'Sinal Aldrin + Dieldrin', 'Sinal Alumínio dissolvido', 'Sinal Alumínio total', 'Sinal Arsênio Dissolvido', 'Sinal Arsênio total', 'Sinal Atrazina', 'Sinal Bário total', 'Sinal Boro dissolvido', 'Sinal Boro total', 'Sinal Cádmio total', 'Sinal Cálcio total', 'Sinal Chumbo total', 'Sinal Cianeto Livre', 'Sinal Cianeto total', 'Sinal Clordano - cis + trans', 'Sinal Cloreto total', 'Sinal Clorofila a', 'Sinal Cobre dissolvido', 'Sinal Cobre total', 'Sinal Coliformes termotolerantes', 'Sinal Coliformes totais', 'Sinal Condição de tempo', 'Sinal Condutividade elétrica in loco', 'Sinal Condutividade elétrica laboratório', 'Sinal Cor verdadeira', 'Sinal Cromo hexavalente', 'Sinal Cromo total', 'Sinal Cromo trivalente', 'Sinal DDT', 'Sinal Demanda Bioquímica de Oxigênio', 'Sinal Demanda Química de Oxigênio', 'Sinal Densidade de cianobactérias', 'Sinal Descarga Liquida', 'Sinal Dureza de Cálcio', 'Sin

In [ ]:
colunas_valor = [
    c for c in df.columns
    if not c.startswith("Sinal")
]

In [ ]:
# Verificar valores ausentes
missing = df.isna().sum()

missing_percent = (
    df.isna().mean() * 100
).sort_values(ascending=False)

auditoria_missing = pd.DataFrame({
    "missing": missing,
    "missing_percent": missing_percent
})

auditoria_missing.head(20)

,missing,missing_percent
"2,4,6 Triclorofenol",39472,99.639026
Alcalinidade de bicarbonato,21345,53.881106
Alcalinidade total,20865,52.669443
Aldrin + Dieldrin,39473,99.641550
Alumínio dissolvido,25225,63.675375
Alumínio total,37841,95.521898
Arsênio Dissolvido,37989,95.895494
Arsênio total,16421,41.451470
Atrazina,39469,99.631453
Boro dissolvido,35812,90.400101


In [ ]:
# Valores ausentes por parâmetro
parametros = [
    "pH in loco",
    "Turbidez",
    "Oxigênio dissolvido",
    "Temperatura da água",
    "Demanda Bioquímica de Oxigênio",
    "Fósforo total",
    "Nitrato"
]

for parametro in parametros:
    percentual = df[parametro].isna().mean() * 100
    print(parametro, round(percentual, 2))

pH in loco 0.23
Turbidez 0.15
Oxigênio dissolvido 0.12
Temperatura da água 0.12
Demanda Bioquímica de Oxigênio 0.78
Fósforo total 0.83
Nitrato 0.86


In [ ]:
# Verificar duplicidade
chave = [
    "Estação",
    "Data de Amostragem",
    "Hora de Amostragem"
]

duplicados = df.duplicated(
    subset=chave,
    keep=False
)

df[duplicados].sort_values(chave)

,Estação,Data de Amostragem,Hora de Amostragem,"Sinal 2,4,6 Triclorofenol","2,4,6 Triclorofenol",Sinal Alcalinidade de bicarbonato,Alcalinidade de bicarbonato,Sinal Alcalinidade total,Alcalinidade total,Sinal Aldrin + Dieldrin,...,Transparência,Sinal Trifluoralina,Trifluoralina,Sinal Turbidez,Turbidez,Sinal Vanádio total,Vanádio total,Sinal Zinco total,Zinco total,ano


In [ ]:
print("Registros duplicados:", duplicados.sum())

Registros duplicados: 0


In [ ]:
duplicados_linha = df.duplicated(keep=False)

print("Linhas completamente duplicadas:", duplicados_linha.sum())

Linhas completamente duplicadas: 0


In [ ]:
# Verificar tipos de dados
pd.options.display.max_rows = None
df.dtypes

,0
Estação,object
Data de Amostragem,datetime64[ns]
Hora de Amostragem,object
"Sinal 2,4,6 Triclorofenol",object
"2,4,6 Triclorofenol",float64
Sinal Alcalinidade de bicarbonato,object
Alcalinidade de bicarbonato,float64
Sinal Alcalinidade total,object
Alcalinidade total,float64
Sinal Aldrin + Dieldrin,object


In [ ]:
# Verificar sinais < e >
for coluna in colunas_sinal:

    valores = df[coluna].dropna().unique()

    print(f"\n{coluna}")
    print(valores)


Sinal 2,4,6 Triclorofenol
['<']

Sinal Alcalinidade de bicarbonato
['<']

Sinal Alcalinidade total
['<']

Sinal Aldrin + Dieldrin
['<']

Sinal Alumínio dissolvido
['<']

Sinal Alumínio total
['<']

Sinal Arsênio Dissolvido
['<']

Sinal Arsênio total
['<']

Sinal Atrazina
['<']

Sinal Bário total
['<']

Sinal Boro dissolvido
['<']

Sinal Boro total
['<']

Sinal Cádmio total
['<']

Sinal Cálcio total
['<']

Sinal Chumbo total
['<']

Sinal Cianeto Livre
['<']

Sinal Cianeto total
['<']

Sinal Clordano - cis + trans
['<']

Sinal Cloreto total
['<']

Sinal Clorofila a
['<']

Sinal Cobre dissolvido
['<']

Sinal Cobre total
['<']

Sinal Coliformes termotolerantes
['<' '>']

Sinal Coliformes totais
['>' '<']

Sinal Condição de tempo
[]

Sinal Condutividade elétrica in loco
[]

Sinal Condutividade elétrica laboratório
[]

Sinal Cor verdadeira
['<']

Sinal Cromo hexavalente
['<']

Sinal Cromo total
['<']

Sinal Cromo trivalente
['<']

Sinal DDT
['<']

Sinal Demanda Bioquímica de Oxigênio
['<']


In [ ]:
for coluna in colunas_sinal:

    print(
        coluna,
        df[coluna].value_counts(dropna=False)
    )

Sinal 2,4,6 Triclorofenol Sinal 2,4,6 Triclorofenol
NaN    39472
<        143
Name: count, dtype: int64
Sinal Alcalinidade de bicarbonato Sinal Alcalinidade de bicarbonato
NaN    39601
<         14
Name: count, dtype: int64
Sinal Alcalinidade total Sinal Alcalinidade total
NaN    39601
<         14
Name: count, dtype: int64
Sinal Aldrin + Dieldrin Sinal Aldrin + Dieldrin
NaN    39473
<        142
Name: count, dtype: int64
Sinal Alumínio dissolvido Sinal Alumínio dissolvido
NaN    30431
<       9184
Name: count, dtype: int64
Sinal Alumínio total Sinal Alumínio total
NaN    39577
<         38
Name: count, dtype: int64
Sinal Arsênio Dissolvido Sinal Arsênio Dissolvido
NaN    39079
<        536
Name: count, dtype: int64
Sinal Arsênio total Sinal Arsênio total
NaN    22571
<      17044
Name: count, dtype: int64
Sinal Atrazina Sinal Atrazina
NaN    39469
<        146
Name: count, dtype: int64
Sinal Bário total Sinal Bário total
NaN    37936
<       1679
Name: count, dtype: int64
Sinal Boro d

In [ ]:
# Verificar valores impossíveis
df["pH in loco"].describe()

,pH in loco
count,39525.000000
mean,6.903609
std,0.592999
min,2.900000
25%,6.500000
50%,6.900000
75%,7.300000
max,11.400000


In [ ]:
df[
    (df["pH in loco"] < 0) |
    (df["pH in loco"] > 14)
]

,Estação,Data de Amostragem,Hora de Amostragem,"Sinal 2,4,6 Triclorofenol","2,4,6 Triclorofenol",Sinal Alcalinidade de bicarbonato,Alcalinidade de bicarbonato,Sinal Alcalinidade total,Alcalinidade total,Sinal Aldrin + Dieldrin,...,Transparência,Sinal Trifluoralina,Trifluoralina,Sinal Turbidez,Turbidez,Sinal Vanádio total,Vanádio total,Sinal Zinco total,Zinco total,ano


In [ ]:
# Parâmetros que não podem ser negativos'
df[df["Turbidez"] < 0]

,Estação,Data de Amostragem,Hora de Amostragem,"Sinal 2,4,6 Triclorofenol","2,4,6 Triclorofenol",Sinal Alcalinidade de bicarbonato,Alcalinidade de bicarbonato,Sinal Alcalinidade total,Alcalinidade total,Sinal Aldrin + Dieldrin,...,Transparência,Sinal Trifluoralina,Trifluoralina,Sinal Turbidez,Turbidez,Sinal Vanádio total,Vanádio total,Sinal Zinco total,Zinco total,ano


In [ ]:
df[df["Temperatura da água"] < 0]

,Estação,Data de Amostragem,Hora de Amostragem,"Sinal 2,4,6 Triclorofenol","2,4,6 Triclorofenol",Sinal Alcalinidade de bicarbonato,Alcalinidade de bicarbonato,Sinal Alcalinidade total,Alcalinidade total,Sinal Aldrin + Dieldrin,...,Transparência,Sinal Trifluoralina,Trifluoralina,Sinal Turbidez,Turbidez,Sinal Vanádio total,Vanádio total,Sinal Zinco total,Zinco total,ano


In [ ]:
# Identificação de valores extremos pelo método do IQR
#
# O IQR é utilizado como mecanismo de detecção de candidatos a valores extremos,
# e não como regra automática de exclusão.

outliers_iqr = {}

for parametro in parametros:
    serie = df[parametro].dropna()

    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    outliers = serie[j
        (serie < limite_inferior) |
        (serie > limite_superior)
    ]

    outliers_iqr[parametro] = {
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "limite_inferior": limite_inferior,
        "limite_superior": limite_superior,
        "n_outliers": len(outliers),
        "percentual_outliers": len(outliers) / len(serie) * 100
    }

auditoria_outliers = pd.DataFrame(outliers_iqr).T

auditoria_outliers.sort_values(
    "percentual_outliers",
    ascending=False
)

,Q1,Q3,IQR,limite_inferior,limite_superior,n_outliers,percentual_outliers
Demanda Bioquímica de Oxigênio,2.00,2.80,0.80,0.800,4.000,7280.0,18.521817
Nitrogênio amoniacal total,0.10,0.30,0.20,-0.200,0.600,6327.0,16.486867
Coliformes termotolerantes,220.00,13000.00,12780.00,-18950.000,32170.000,3219.0,14.617201
Turbidez,8.11,50.50,42.39,-55.475,114.085,5318.0,13.444231
Fósforo total,0.02,0.12,0.10,-0.130,0.270,4442.0,11.306251
Condutividade elétrica in loco,38.70,134.00,95.30,-104.250,276.950,4360.0,11.048884
Oxigênio dissolvido,6.30,7.80,1.50,4.050,10.050,4066.0,10.276240
Nitrato,0.11,0.68,0.57,-0.745,1.535,3189.0,8.119462
Sólidos totais,57.00,180.00,123.00,-127.500,364.500,3097.0,7.947547
pH in loco,6.50,7.30,0.80,5.300,8.500,625.0,1.581278


In [ ]:
# Estatísticas descritivas dos parâmetros selecionados
#
# Conjunto reduzido de parâmetros para a auditoria exploratória.
#
# Critérios de seleção:
# 1. relevância para a caracterização da qualidade da água;
# 2. relação com parâmetros utilizados pelo IGAM no IQA;
# 3. elevada cobertura na série histórica;
# 4. diversidade de dimensões físico-químicas e microbiológicas;
# 5. tamanho reduzido para facilitar a demonstração do sistema.
#
# A seleção não representa todos os parâmetros disponíveis na base.
# Os demais parâmetros serão preservados e poderão ser analisados
# posteriormente conforme a pergunta e a disponibilidade dos dados.

parametros = [
    "pH in loco",
    "Turbidez",
    "Oxigênio dissolvido",
    "Temperatura da água",
    "Condutividade elétrica in loco",
    "Demanda Bioquímica de Oxigênio",
    "Fósforo total",
    "Nitrato",
    "Sólidos totais",
    "Nitrogênio amoniacal total",
    "Coliformes termotolerantes"
]

estatisticas = df[parametros].describe().T

estatisticas




,count,mean,std,min,25%,50%,75%,max
pH in loco,39525.0,6.903609,0.592999,2.900,6.50,6.90,7.30,11.40
Turbidez,39556.0,84.677701,339.992420,0.290,8.11,18.60,50.50,17949.00
Oxigênio dissolvido,39567.0,6.736790,1.833971,0.200,6.30,7.20,7.80,18.60
Temperatura da água,39566.0,23.915248,3.461307,0.140,21.70,24.10,26.30,39.80
Condutividade elétrica in loco,39461.0,124.289372,185.582486,0.450,38.70,62.80,134.00,3800.00
Demanda Bioquímica de Oxigênio,39305.0,5.706409,20.042314,0.100,2.00,2.00,2.80,921.00
Fósforo total,39288.0,0.139464,0.297831,0.010,0.02,0.05,0.12,9.24
Nitrato,39276.0,0.597732,1.073309,0.002,0.11,0.28,0.68,34.70
Sólidos totais,38968.0,168.111856,292.111055,6.000,57.00,94.00,180.00,11490.00
Nitrogênio amoniacal total,38376.0,1.119940,3.909856,0.020,0.10,0.11,0.30,99.70


In [ ]:
# Cobertura dos parâmetros

total_registros = len(df)

auditoria_parametros = estatisticas.copy()

auditoria_parametros["missing"] = (
    total_registros - auditoria_parametros["count"]
)

auditoria_parametros["cobertura_%"] = (
    auditoria_parametros["count"] / total_registros * 100
)

auditoria_parametros["missing_%"] = (
    auditoria_parametros["missing"] / total_registros * 100
)

auditoria_parametros[
    ["count", "missing", "cobertura_%", "missing_%"]
].sort_values(
    "cobertura_%"
)

,count,missing,cobertura_%,missing_%
Coliformes termotolerantes,22022.0,17593.0,55.590054,44.409946
Nitrogênio amoniacal total,38376.0,1239.0,96.872397,3.127603
Sólidos totais,38968.0,647.0,98.366780,1.633220
Nitrato,39276.0,339.0,99.144264,0.855736
Fósforo total,39288.0,327.0,99.174555,0.825445
Demanda Bioquímica de Oxigênio,39305.0,310.0,99.217468,0.782532
Condutividade elétrica in loco,39461.0,154.0,99.611258,0.388742
pH in loco,39525.0,90.0,99.772813,0.227187
Turbidez,39556.0,59.0,99.851067,0.148933
Temperatura da água,39566.0,49.0,99.876309,0.123691


Os parâmetros selecionados apresentam elevada cobertura na série histórica, com exceção de coliformes termotolerantes, que possui 55,59% de cobertura. A diferença de completude deve ser considerada antes de análises que dependam desse parâmetro. A ausência dos dados não será tratada automaticamente como erro ou imputada nesta etapa.

In [ ]:
# Análise de valores ausentes por ano para todos os parâmetros

df["ano"] = df["Data de Amostragem"].dt.year

ausencia_por_ano = (
    df.groupby("ano")[parametros]
      .apply(lambda grupo: grupo.isna().mean() * 100)
)

ausencia_por_ano

,pH in loco,Turbidez,Oxigênio dissolvido,Temperatura da água,Condutividade elétrica in loco,Demanda Bioquímica de Oxigênio,Fósforo total,Nitrato,Sólidos totais,Nitrogênio amoniacal total,Coliformes termotolerantes
ano,,,,,,,,,,,
1997,0.000000,0.000000,0.000000,0.000000,10.688836,1.187648,4.038005,0.712589,23.752969,0.000000,0.000000
1998,0.000000,0.000000,0.000000,0.000000,0.000000,0.671141,0.000000,0.000000,0.000000,0.000000,0.805369
1999,0.000000,0.000000,0.000000,0.000000,0.000000,0.251572,0.000000,0.000000,0.000000,0.000000,0.125786
2000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2001,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.103950,0.000000,0.000000,1.039501
2002,0.000000,0.000000,0.000000,0.000000,0.000000,1.338826,1.235839,1.235839,0.000000,1.235839,0.411946
2003,0.000000,0.086281,0.000000,0.000000,0.000000,4.831752,4.831752,4.831752,0.172563,4.831752,1.294219
2004,0.000000,0.086133,0.000000,0.000000,0.000000,4.823428,4.823428,4.823428,0.086133,4.823428,1.119724
2005,0.000000,0.000000,0.086505,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.276817


In [ ]:
# Maior percentual de ausência observado por parâmetro

maior_ausencia = pd.DataFrame({
    "maior_missing_%": ausencia_por_ano.max(),
    "ano_maior_missing": ausencia_por_ano.idxmax()
})

maior_ausencia.sort_values(
    "maior_missing_%",
    ascending=False
)

,maior_missing_%,ano_maior_missing
Coliformes termotolerantes,100.000000,2014
Sólidos totais,23.752969,1997
Nitrogênio amoniacal total,11.484185,2009
Condutividade elétrica in loco,10.688836,1997
Nitrato,5.673222,2006
Fósforo total,4.831752,2003
Demanda Bioquímica de Oxigênio,4.831752,2003
Turbidez,1.420765,2018
Temperatura da água,1.384335,2018
Oxigênio dissolvido,1.275046,2018


In [ ]:
# Quantidade de registros por ano

registros_por_ano = (
    df["ano"]
    .value_counts()
    .sort_index()
)

registros_por_ano

,count
ano,
1997,421
1998,745
1999,795
2000,959
2001,962
2002,971
2003,1159
2004,1161
2005,1156


In [ ]:
# Verificar os registros de 2014

df_2014 = df[df["ano"] == 2014]

print("Registros em 2014:", len(df_2014))
print(
    "Coliformes preenchidos:",
    df_2014["Coliformes termotolerantes"].notna().sum()
)
print(
    "Coliformes ausentes:",
    df_2014["Coliformes termotolerantes"].isna().sum()
)

Registros em 2014: 2637
Coliformes preenchidos: 0
Coliformes ausentes: 2637


O parâmetro Coliformes termotolerantes apresenta ausência de dados em 100% dos 2.637 registros referentes ao ano de 2014. A causa dessa ausência não é determinada pela base analisada e requer investigação adicional.

In [ ]:
# Coliformes termotolerantes em 2014 por estação

coliformes_2014_estacao = (
    df_2014.groupby("Estação")["Coliformes termotolerantes"]
    .agg(
        registros="size",
        preenchidos="count"
    )
)

coliformes_2014_estacao["ausentes"] = (
    coliformes_2014_estacao["registros"]
    - coliformes_2014_estacao["preenchidos"]
)

coliformes_2014_estacao["missing_%"] = (
    coliformes_2014_estacao["ausentes"]
    / coliformes_2014_estacao["registros"]
    * 100
)

coliformes_2014_estacao

,registros,preenchidos,ausentes,missing_%
Estação,,,,
AV007,4,0,4,100.0
AV010,4,0,4,100.0
AV020,4,0,4,100.0
AV050,4,0,4,100.0
AV060,4,0,4,100.0
AV070,4,0,4,100.0
AV080,4,0,4,100.0
AV120,4,0,4,100.0
AV160E,2,0,2,100.0


In [ ]:
# Verificar ausência sistemática de valores por ano
(ausencia_por_ano == 100).sum()

,0
pH in loco,0
Turbidez,0
Oxigênio dissolvido,0
Temperatura da água,0
Condutividade elétrica in loco,0
Demanda Bioquímica de Oxigênio,0
Fósforo total,0
Nitrato,0
Sólidos totais,0
Nitrogênio amoniacal total,0


In [ ]:
# Anos em que Coliformes termotolerantes
# apresentam 100% de valores ausentes

anos_100_missing = ausencia_por_ano[
    ausencia_por_ano["Coliformes termotolerantes"] == 100
].index.tolist()

anos_100_missing

[2014, 2015, 2016, 2017, 2018, 2019]

In [ ]:
# Quantidade de registros nos anos com 100% de ausência

registros_anos_100 = (
    df[df["ano"].isin(anos_100_missing)]
    .groupby("ano")
    .size()
    .reset_index(name="registros")
)

registros_anos_100

,ano,registros
0,2014,2637
1,2015,2626
2,2016,2542
3,2017,1734
4,2018,2745
5,2019,2786


Entre 2014 e 2019, o parâmetro Coliformes termotolerantes apresenta ausência completa na base analisada. São 15.070 registros distribuídos nesses seis anos, sem nenhuma observação preenchida para o parâmetro.

In [ ]:
# Cobertura de Coliformes termotolerantes por estação
# no período de 2014 a 2019

periodo_2014_2019 = df[
    df["ano"].between(2014, 2019)
]

cobertura_coliformes_estacao = (
    periodo_2014_2019
    .groupby("Estação")["Coliformes termotolerantes"]
    .agg(
        registros="size",
        preenchidos="count"
    )
)

cobertura_coliformes_estacao["ausentes"] = (
    cobertura_coliformes_estacao["registros"]
    - cobertura_coliformes_estacao["preenchidos"]
)

cobertura_coliformes_estacao["cobertura_%"] = (
    cobertura_coliformes_estacao["preenchidos"]
    / cobertura_coliformes_estacao["registros"]
    * 100
)

cobertura_coliformes_estacao.sort_values(
    "cobertura_%"
)

,registros,preenchidos,ausentes,cobertura_%
Estação,,,,
VG001,16,0,16,0.0
UR018,7,0,7,0.0
UR017,21,0,21,0.0
UR016,22,0,22,0.0
UR015,22,0,22,0.0
UR014,22,0,22,0.0
UR013,22,0,22,0.0
UR012,22,0,22,0.0
UR011,22,0,22,0.0


Entre 2014 e 2019, nenhuma das estações apresenta registros preenchidos para Coliformes termotolerantes.

Portanto, notou-se uma ausência sistemática de registros desse parâmetro.

- Parâmetro: Coliformes termotolerantes
- Período: 2014–2019
- Estações afetadas: todas as estações presentes no período
- Cobertura no período: 0%
- Registros no período: 15.070
- Registros preenchidos: 0
- Registros ausentes: 15.070

In [ ]:
# Identificar períodos com ausência total para cada parâmetro

anos_100_missing_por_parametro = {}

for parametro in parametros:
    anos = ausencia_por_ano[
        ausencia_por_ano[parametro] == 100
    ].index.tolist()

    anos_100_missing_por_parametro[parametro] = anos

anos_100_missing_por_parametro

{'pH in loco': [],
 'Turbidez': [],
 'Oxigênio dissolvido': [],
 'Temperatura da água': [],
 'Condutividade elétrica in loco': [],
 'Demanda Bioquímica de Oxigênio': [],
 'Fósforo total': [],
 'Nitrato': [],
 'Sólidos totais': [],
 'Nitrogênio amoniacal total': [],
 'Coliformes termotolerantes': [2014, 2015, 2016, 2017, 2018, 2019]}

In [ ]:
# Queremos saber se as estações apresentam uma frequência de coleta relativamente consistente
# ao longo dos anos ou existem períodos com poucas/nenhuma observação
#
# Quantidade de amostras por estação e ano
amostras_estacao_ano = (
    df.groupby(["Estação", "ano"])
      .size()
      .reset_index(name="n_amostras")
)

amostras_estacao_ano.head(10)

,Estação,ano,n_amostras
0,AV005,2003,4
1,AV005,2004,4
2,AV005,2006,1
3,AV005,2007,4
4,AV005,2008,4
5,AV005,2009,4
6,AV005,2010,4
7,AV005,2011,4
8,AV005,2012,4
9,AV005,2013,3


In [ ]:
amostras_estacao_ano["n_amostras"].describe()

,n_amostras
count,9883.000000
mean,4.008398
std,1.565164
min,1.000000
25%,4.000000
50%,4.000000
75%,4.000000
max,12.000000


In [ ]:
# Verificar quantas estações estão representadas nos registros
print("Número de estações:", amostras_estacao_ano["Estação"].nunique())

print("\nNúmero de anos:", amostras_estacao_ano["ano"].nunique())

print("\nDistribuição da quantidade de amostras por estação-ano:")
print(amostras_estacao_ano["n_amostras"].value_counts().sort_index())

Número de estações: 769

Número de anos: 23

Distribuição da quantidade de amostras por estação-ano:
n_amostras
1      202
2      629
3      746
4     7879
5       15
6       28
7       14
8       56
9       33
10      13
11      31
12     237
Name: count, dtype: int64


Das 9.883 combinações estação–ano, 7.879 têm exatamente 4 amostras, aproximadamente 79,7%. Portanto, 4 amostras por ano é o padrão predominante.
Isso sugere que a base possui uma estrutura temporal predominantemente regular, provavelmente compatível com uma frequência trimestral em grande parte das combinações, porém cabe realizar uma investigação mais detalhada.

In [ ]:
# Separar casos de baixa frequência
baixa_frequencia = (
    amostras_estacao_ano[
        amostras_estacao_ano["n_amostras"] <= 2
    ]
    .sort_values(["ano", "Estação"])
)

print("Combinações estação-ano com até 2 amostras:",
      len(baixa_frequencia))

baixa_frequencia.head(20)

Combinações estação-ano com até 2 amostras: 831


,Estação,ano,n_amostras
323,BG001,1997,1
349,BG003,1997,2
375,BG005,1997,2
401,BG007,1997,2
433,BG009,1997,2
467,BG011,1997,2
510,BG013,1997,2
553,BG015,1997,2
576,BG017,1997,2
602,BG019,1997,2


In [ ]:
# Separar casos de alta fequência
alta_frequencia = (
    amostras_estacao_ano[
        amostras_estacao_ano["n_amostras"] >= 8
    ]
    .sort_values(["ano", "Estação"])
)

print("Combinações estação-ano com 8 ou mais amostras:",
      len(alta_frequencia))

alta_frequencia.head(20)

Combinações estação-ano com 8 ou mais amostras: 370


,Estação,ano,n_amostras
126,AV090,2003,12
129,AV100,2003,12
132,AV120,2003,12
148,AV140,2003,12
151,AV150,2003,12
272,AV300,2003,12
289,AV320,2003,12
127,AV090,2004,12
130,AV100,2004,12
133,AV120,2004,12


In [ ]:
# Distribuição dos casos de baixa frequência por ano
baixa_por_ano = (
    baixa_frequencia
    .groupby("ano")
    .size()
    .reset_index(name="combinacoes_ate_2_amostras")
)

print(baixa_por_ano)

     ano  combinacoes_ate_2_amostras
0   1997                         183
1   1998                           5
2   1999                           3
3   2000                           2
4   2001                           2
5   2002                           9
6   2004                           1
7   2005                          37
8   2006                          54
9   2007                          56
10  2008                          55
11  2009                          30
12  2010                          57
13  2011                          44
14  2012                          38
15  2013                          22
16  2014                           5
17  2015                          17
18  2016                          30
19  2017                         157
20  2018                           9
21  2019                          15


In [ ]:
# Distribuição dos casos de alta frequência por ano
alta_por_ano = (
    alta_frequencia
    .groupby("ano")
    .size()
    .reset_index(name="combinacoes_8_ou_mais_amostras")
)

print(alta_por_ano)

     ano  combinacoes_8_ou_mais_amostras
0   2003                               7
1   2004                               7
2   2007                               4
3   2008                              19
4   2009                              22
5   2010                              22
6   2011                              21
7   2012                              21
8   2013                              37
9   2014                              40
10  2015                              46
11  2016                              46
12  2017                              15
13  2018                              30
14  2019                              33


Foram identificadas 831 combinações com até 2 registros e 370 com 8 ou mais registros. As diferenças de frequência apresentam concentração em determinados anos, especialmente nos primeiros anos da série e em 2017 para baixa frequência, e entre 2013–2016 para alta frequência.

Logo, a frequência de amostragem não é homogênea ao longo da série histórica. A maior parte das combinações estação–ano possui 4 observações, mas há períodos com menor e maior frequência, com concentração temporal desses desvios. Essas diferenças devem ser consideradas na interpretação temporal e comparativa, mas não foram classificadas como erros de qualidade sem evidência adicional.

In [ ]:
# ============================================
# TABELA DE ACHADOS DA AUDITORIA INICIAL
# ============================================

achados_auditoria = pd.DataFrame([
    {
        "id": "AQ-01",
        "categoria": "Estrutura",
        "verificacao": "Estrutura da base principal",
        "resultado": (
            "39.615 registros, 769 estações e período de "
            "02/07/1997 a 12/12/2019."
        ),
        "evidencia": (
            "df.shape; df['Estação'].nunique(); "
            "df['Data de Amostragem'].min()/max()"
        ),
        "severidade": "Informativo",
        "acao_recomendada": "Base apta para análise exploratória inicial."
    },

    {
        "id": "AQ-02",
        "categoria": "Duplicidade",
        "verificacao": "Duplicidade da chave estação-data-hora",
        "resultado": (
            "Não foram identificados registros duplicados pela combinação "
            "Estação + Data de Amostragem + Hora de Amostragem."
        ),
        "evidencia": "Registros duplicados pela chave = 0",
        "severidade": "OK",
        "acao_recomendada": "Manter a chave como controle de duplicidade."
    },

    {
        "id": "AQ-03",
        "categoria": "Completude",
        "verificacao": "Cobertura de Coliformes termotolerantes",
        "resultado": (
            "22.022 registros preenchidos de 39.615; cobertura de "
            "55,59% e ausência de dados em 44,41% dos registros."
        ),
        "evidencia": (
            "Contagem de valores não nulos e cálculo de cobertura "
            "do parâmetro."
        ),
        "severidade": "Atenção",
        "acao_recomendada": (
            "Preservar os dados ausentes e considerar a cobertura "
            "antes de análises que utilizem esse parâmetro."
        )
    },

    {
        "id": "AQ-04",
        "categoria": "Completude temporal",
        "verificacao": "Ausência sistemática de Coliformes termotolerantes",
        "resultado": (
            "Entre 2014 e 2019, foram identificados 15.070 registros "
            "sem observações de Coliformes termotolerantes. "
            "As estações presentes nesses anos apresentam 0% de cobertura."
        ),
        "evidencia": (
            "Análise por ano e por estação; anos com 100% de ausência: "
            "2014, 2015, 2016, 2017, 2018 e 2019."
        ),
        "severidade": "Atenção",
        "acao_recomendada": (
            "Não imputar automaticamente. Investigar a origem da ausência "
            "e bloquear interpretações de tendência contínua para esse parâmetro."
        )
    },

    {
        "id": "AQ-05",
        "categoria": "Consistência",
        "verificacao": "Valores de pH fora do intervalo [0, 14]",
        "resultado": "Nenhum valor fora do intervalo foi identificado.",
        "evidencia": "Teste de consistência aplicado à variável pH.",
        "severidade": "OK",
        "acao_recomendada": "Manter os valores; regra pode ser incorporada ao controle automatizado."
    },

    {
        "id": "AQ-06",
        "categoria": "Consistência",
        "verificacao": "Valores negativos de turbidez",
        "resultado": "Nenhum valor negativo foi identificado.",
        "evidencia": "Teste de consistência aplicado à variável Turbidez.",
        "severidade": "OK",
        "acao_recomendada": "Manter os valores; regra pode ser incorporada ao controle automatizado."
    },

    {
        "id": "AQ-07",
        "categoria": "Consistência",
        "verificacao": "Valores negativos de temperatura da água",
        "resultado": "Nenhum valor negativo foi identificado.",
        "evidencia": "Teste de consistência aplicado à variável Temperatura da água.",
        "severidade": "OK",
        "acao_recomendada": "Manter os valores; regra pode ser incorporada ao controle automatizado."
    },

    {
        "id": "AQ-08",
        "categoria": "Valores extremos",
        "verificacao": "Identificação de possíveis outliers pelo método do IQR",
        "resultado": (
            "Foram identificados valores extremos em diferentes parâmetros. "
            "A frequência de candidatos variou entre os parâmetros."
        ),
        "evidencia": (
            "Método do intervalo interquartil (IQR) aplicado aos 11 "
            "parâmetros selecionados."
        ),
        "severidade": "Atenção",
        "acao_recomendada": (
            "Tratar os outliers como candidatos à investigação, "
            "sem exclusão automática."
        )
    },

    {
        "id": "AQ-09",
        "categoria": "Cobertura amostral",
        "verificacao": "Número de amostras por estação e ano",
        "resultado": (
            "Foram identificadas 9.883 combinações estação-ano. "
            "Em 7.879 delas (aproximadamente 79,7%), houve exatamente "
            "4 amostras no ano."
        ),
        "evidencia": (
            "Distribuição da quantidade de registros por combinação "
            "Estação + Ano."
        ),
        "severidade": "Informativo",
        "acao_recomendada": (
            "Considerar a frequência amostral nas análises temporais "
            "e investigar combinações com frequência muito baixa ou alta."
        )
    },

    {
        "id": "AQ-10",
        "categoria": "Semântica dos dados",
        "verificacao": "Presença de sinais de medição",
        "resultado": (
            "A base possui colunas de sinal associadas aos parâmetros, "
            "incluindo operadores como '<' e '>'."
        ),
        "evidencia": (
            "Colunas 'Sinal ...' e distribuição dos valores das colunas "
            "de sinal."
        ),
        "severidade": "Atenção",
        "acao_recomendada": (
            "Preservar o sinal na transformação dos dados e não interpretar "
            "automaticamente valores censurados como medições exatas."
        )
    }
])

# Visualizar a tabela
display(achados_auditoria)

,id,categoria,verificacao,resultado,evidencia,severidade,acao_recomendada
0,AQ-01,Estrutura,Estrutura da base principal,"39.615 registros, 769 estações e período de 02...",df.shape; df['Estação'].nunique(); df['Data de...,Informativo,Base apta para análise exploratória inicial.
1,AQ-02,Duplicidade,Duplicidade da chave estação-data-hora,Não foram identificados registros duplicados p...,Registros duplicados pela chave = 0,OK,Manter a chave como controle de duplicidade.
2,AQ-03,Completude,Cobertura de Coliformes termotolerantes,22.022 registros preenchidos de 39.615; cobert...,Contagem de valores não nulos e cálculo de cob...,Atenção,Preservar os dados ausentes e considerar a cob...
3,AQ-04,Completude temporal,Ausência sistemática de Coliformes termotolera...,"Entre 2014 e 2019, foram identificados 15.070 ...",Análise por ano e por estação; anos com 100% d...,Atenção,Não imputar automaticamente. Investigar a orig...
4,AQ-05,Consistência,"Valores de pH fora do intervalo [0, 14]",Nenhum valor fora do intervalo foi identificado.,Teste de consistência aplicado à variável pH.,OK,Manter os valores; regra pode ser incorporada ...
5,AQ-06,Consistência,Valores negativos de turbidez,Nenhum valor negativo foi identificado.,Teste de consistência aplicado à variável Turb...,OK,Manter os valores; regra pode ser incorporada ...
6,AQ-07,Consistência,Valores negativos de temperatura da água,Nenhum valor negativo foi identificado.,Teste de consistência aplicado à variável Temp...,OK,Manter os valores; regra pode ser incorporada ...
7,AQ-08,Valores extremos,Identificação de possíveis outliers pelo métod...,Foram identificados valores extremos em difere...,Método do intervalo interquartil (IQR) aplicad...,Atenção,Tratar os outliers como candidatos à investiga...
8,AQ-09,Cobertura amostral,Número de amostras por estação e ano,Foram identificadas 9.883 combinações estação-...,Distribuição da quantidade de registros por co...,Informativo,Considerar a frequência amostral nas análises ...
9,AQ-10,Semântica dos dados,Presença de sinais de medição,A base possui colunas de sinal associadas aos ...,Colunas 'Sinal ...' e distribuição dos valores...,Atenção,Preservar o sinal na transformação dos dados e...


In [ ]:
# Salvar tabela como csv
achados_auditoria.to_csv(
    "achados_auditoria.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo salvo: achados_auditoria.csv")

Arquivo salvo: achados_auditoria.csv


In [ ]:
# Queremos transformar os achados em regras que o sistema poderá
# executar novamente em outra base compatível.
#
# ============================================
# REGRAS DE QUALIDADE DOS DADOS
# ============================================

regras_qualidade = pd.DataFrame([
    {
        "id": "CQ-01",
        "categoria": "Estrutura",
        "regra": "Verificar presença das colunas obrigatórias",
        "criterio": (
            "A base deve conter Estação, Data de Amostragem "
            "e Hora de Amostragem."
        ),
        "tipo": "Bloqueante",
        "acao_falha": (
            "Interromper a análise e informar quais colunas "
            "obrigatórias estão ausentes."
        )
    },

    {
        "id": "CQ-02",
        "categoria": "Estrutura",
        "regra": "Verificar duplicidade da observação",
        "criterio": (
            "Não deve haver duplicidade na combinação "
            "Estação + Data de Amostragem + Hora de Amostragem."
        ),
        "tipo": "Atenção",
        "acao_falha": (
            "Sinalizar registros duplicados para investigação; "
            "não excluir automaticamente."
        )
    },

    {
        "id": "CQ-03",
        "categoria": "Completude",
        "regra": "Calcular cobertura dos parâmetros",
        "criterio": (
            "Calcular quantidade e percentual de valores "
            "preenchidos e ausentes para cada parâmetro."
        ),
        "tipo": "Informativa",
        "acao_falha": (
            "Registrar a cobertura e sinalizar parâmetros "
            "com baixa disponibilidade de dados."
        )
    },

    {
        "id": "CQ-04",
        "categoria": "Completude temporal",
        "regra": "Detectar ausência sistemática de dados",
        "criterio": (
            "Identificar parâmetros com períodos em que a "
            "ausência de dados seja sistemática por ano "
            "ou outro intervalo temporal."
        ),
        "tipo": "Atenção",
        "acao_falha": (
            "Sinalizar o período afetado e impedir interpretações "
            "de continuidade temporal sem evidência suficiente."
        )
    },

    {
        "id": "CQ-05",
        "categoria": "Consistência",
        "regra": "Verificar regras físicas previamente declaradas",
        "criterio": (
            "Aplicar somente restrições de domínio explicitamente "
            "definidas para cada variável."
        ),
        "tipo": "Atenção",
        "acao_falha": (
            "Sinalizar valores incompatíveis com a regra, "
            "sem exclusão automática."
        )
    },

    {
        "id": "CQ-06",
        "categoria": "Valores extremos",
        "regra": "Identificar possíveis outliers",
        "criterio": (
            "Utilizar o intervalo interquartil (IQR) como método "
            "de triagem de valores extremos."
        ),
        "tipo": "Diagnóstica",
        "acao_falha": (
            "Sinalizar observações extremas para investigação. "
            "Não considerar o outlier como erro automaticamente."
        )
    },

    {
        "id": "CQ-07",
        "categoria": "Cobertura amostral",
        "regra": "Verificar frequência de amostragem",
        "criterio": (
            "Calcular o número de observações por estação e "
            "período temporal."
        ),
        "tipo": "Diagnóstica",
        "acao_falha": (
            "Sinalizar estações/períodos com frequência "
            "amostral muito baixa ou discrepante."
        )
    },

    {
        "id": "CQ-08",
        "categoria": "Semântica",
        "regra": "Preservar sinais associados às medições",
        "criterio": (
            "Manter os campos de sinal, como '<' e '>', "
            "associados aos respectivos valores."
        ),
        "tipo": "Bloqueante",
        "acao_falha": (
            "Impedir a transformação que descarte o sinal "
            "ou altere sua interpretação."
        )
    },

    {
        "id": "CQ-09",
        "categoria": "Dados ausentes",
        "regra": "Não imputar valores automaticamente",
        "criterio": (
            "Valores ausentes devem permanecer ausentes durante "
            "a auditoria, salvo quando uma estratégia de imputação "
            "for explicitamente definida para uma análise."
        ),
        "tipo": "Bloqueante",
        "acao_falha": (
            "Manter NaN e registrar a ausência no relatório."
        )
    }
])

display(regras_qualidade)

,id,categoria,regra,criterio,tipo,acao_falha
0,CQ-01,Estrutura,Verificar presença das colunas obrigatórias,"A base deve conter Estação, Data de Amostragem...",Bloqueante,Interromper a análise e informar quais colunas...
1,CQ-02,Estrutura,Verificar duplicidade da observação,Não deve haver duplicidade na combinação Estaç...,Atenção,Sinalizar registros duplicados para investigaç...
2,CQ-03,Completude,Calcular cobertura dos parâmetros,Calcular quantidade e percentual de valores pr...,Informativa,Registrar a cobertura e sinalizar parâmetros c...
3,CQ-04,Completude temporal,Detectar ausência sistemática de dados,Identificar parâmetros com períodos em que a a...,Atenção,Sinalizar o período afetado e impedir interpre...
4,CQ-05,Consistência,Verificar regras físicas previamente declaradas,Aplicar somente restrições de domínio explicit...,Atenção,"Sinalizar valores incompatíveis com a regra, s..."
5,CQ-06,Valores extremos,Identificar possíveis outliers,Utilizar o intervalo interquartil (IQR) como m...,Diagnóstica,Sinalizar observações extremas para investigaç...
6,CQ-07,Cobertura amostral,Verificar frequência de amostragem,Calcular o número de observações por estação e...,Diagnóstica,Sinalizar estações/períodos com frequência amo...
7,CQ-08,Semântica,Preservar sinais associados às medições,"Manter os campos de sinal, como '<' e '>', ass...",Bloqueante,Impedir a transformação que descarte o sinal o...
8,CQ-09,Dados ausentes,Não imputar valores automaticamente,Valores ausentes devem permanecer ausentes dur...,Bloqueante,Manter NaN e registrar a ausência no relatório.


In [ ]:
print(f"Total de regras: {len(regras_qualidade)}")

print("\nRegras por categoria:")
display(regras_qualidade["categoria"].value_counts())

print("\nRegras por tipo:")
display(regras_qualidade["tipo"].value_counts())

Total de regras: 9

Regras por categoria:


,count
categoria,
Estrutura,2
Completude,1
Completude temporal,1
Consistência,1
Valores extremos,1
Cobertura amostral,1
Semântica,1
Dados ausentes,1



Regras por tipo:


,count
tipo,
Bloqueante,3
Atenção,3
Diagnóstica,2
Informativa,1


In [ ]:
# Salvar como cvs
regras_qualidade.to_csv(
    "regras_qualidade.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo salvo: regras_qualidade.csv")

Arquivo salvo: regras_qualidade.csv


In [ ]:
# Criar a função do quality_checker
# ============================================
# QUALITY CHECKER - VERSÃO 1
# Regras CQ-01, CQ-02 e CQ-03
# ============================================

def executar_quality_checker(df):

    resultados = []

    # ----------------------------------------
    # CQ-01 - Estrutura
    # ----------------------------------------

    colunas_obrigatorias = [
        "Estação",
        "Data de Amostragem",
        "Hora de Amostragem"
    ]

    colunas_ausentes = [
        coluna for coluna in colunas_obrigatorias
        if coluna not in df.columns
    ]

    if len(colunas_ausentes) == 0:
        resultados.append({
            "regra": "CQ-01",
            "status": "OK",
            "resultado": "Todas as colunas obrigatórias estão presentes.",
            "evidencia": ", ".join(colunas_obrigatorias)
        })
    else:
        resultados.append({
            "regra": "CQ-01",
            "status": "BLOQUEADO",
            "resultado": "Existem colunas obrigatórias ausentes.",
            "evidencia": ", ".join(colunas_ausentes)
        })


    # ----------------------------------------
    # CQ-02 - Duplicidade
    # ----------------------------------------

    chave = [
        "Estação",
        "Data de Amostragem",
        "Hora de Amostragem"
    ]

    duplicados = df.duplicated(
        subset=chave,
        keep=False
    ).sum()

    if duplicados == 0:
        resultados.append({
            "regra": "CQ-02",
            "status": "OK",
            "resultado": "Nenhuma duplicidade identificada.",
            "evidencia": "0 registros duplicados pela chave estação-data-hora."
        })
    else:
        resultados.append({
            "regra": "CQ-02",
            "status": "ATENÇÃO",
            "resultado": f"{duplicados} registros envolvidos em duplicidades.",
            "evidencia": "Duplicidade detectada na chave estação-data-hora."
        })


    # ----------------------------------------
    # CQ-03 - Completude
    # ----------------------------------------

    parametros = [
        "pH in loco",
        "Turbidez",
        "Oxigênio dissolvido",
        "Temperatura da água",
        "Condutividade elétrica in loco",
        "Demanda Bioquímica de Oxigênio",
        "Fósforo total",
        "Nitrato",
        "Sólidos totais",
        "Nitrogênio amoniacal total",
        "Coliformes termotolerantes"
    ]

    for parametro in parametros:

        if parametro not in df.columns:
            resultados.append({
                "regra": "CQ-03",
                "status": "ATENÇÃO",
                "resultado": f"Parâmetro ausente: {parametro}",
                "evidencia": "Coluna não encontrada na base."
            })
            continue

        total = len(df)
        preenchidos = df[parametro].notna().sum()
        ausentes = df[parametro].isna().sum()
        cobertura = (preenchidos / total) * 100

        resultados.append({
            "regra": "CQ-03",
            "status": "OK",
            "resultado": (
                f"{parametro}: {cobertura:.2f}% de cobertura."
            ),
            "evidencia": (
                f"Preenchidos={preenchidos}; "
                f"ausentes={ausentes}; "
                f"total={total}."
            )
        })


    return pd.DataFrame(resultados)

In [ ]:
resultado_quality_checker = executar_quality_checker(df)

display(resultado_quality_checker)

,regra,status,resultado,evidencia
0,CQ-01,OK,Todas as colunas obrigatórias estão presentes.,"Estação, Data de Amostragem, Hora de Amostragem"
1,CQ-02,OK,Nenhuma duplicidade identificada.,0 registros duplicados pela chave estação-data...
2,CQ-03,OK,pH in loco: 99.77% de cobertura.,Preenchidos=39525; ausentes=90; total=39615.
3,CQ-03,OK,Turbidez: 99.85% de cobertura.,Preenchidos=39556; ausentes=59; total=39615.
4,CQ-03,OK,Oxigênio dissolvido: 99.88% de cobertura.,Preenchidos=39567; ausentes=48; total=39615.
5,CQ-03,OK,Temperatura da água: 99.88% de cobertura.,Preenchidos=39566; ausentes=49; total=39615.
6,CQ-03,OK,Condutividade elétrica in loco: 99.61% de cobe...,Preenchidos=39461; ausentes=154; total=39615.
7,CQ-03,OK,Demanda Bioquímica de Oxigênio: 99.22% de cobe...,Preenchidos=39305; ausentes=310; total=39615.
8,CQ-03,OK,Fósforo total: 99.17% de cobertura.,Preenchidos=39288; ausentes=327; total=39615.
9,CQ-03,OK,Nitrato: 99.14% de cobertura.,Preenchidos=39276; ausentes=339; total=39615.


In [ ]:
# Criar cópia para testar o funcionamento do checker
df_teste = df.copy()

In [ ]:
# Adiciona uma cópia do primeiro registro
df_teste = pd.concat(
    [df_teste, df_teste.iloc[[0]]],
    ignore_index=True
)

print("Registros originais:", len(df))
print("Registros após inserção do teste:", len(df_teste))

Registros originais: 39615
Registros após inserção do teste: 39616


In [ ]:
resultado_teste = executar_quality_checker(df_teste)

display(
    resultado_teste[
        resultado_teste["regra"] == "CQ-02"
    ]
)

,regra,status,resultado,evidencia
1,CQ-02,ATENÇÃO,2 registros envolvidos em duplicidades.,Duplicidade detectada na chave estação-data-hora.


In [ ]:
# ============================================
# CQ-04 - AUSÊNCIA SISTEMÁTICA DE DADOS
# ============================================

def verificar_ausencia_temporal(df, parametros):

    resultados = []

    # Criar ano a partir da data de amostragem
    dados = df.copy()
    dados["ano"] = pd.to_datetime(
        dados["Data de Amostragem"],
        errors="coerce"
    ).dt.year

    for parametro in parametros:

        if parametro not in dados.columns:
            resultados.append({
                "regra": "CQ-04",
                "parametro": parametro,
                "ano": None,
                "registros": None,
                "preenchidos": None,
                "ausentes": None,
                "cobertura_pct": None,
                "status": "ATENÇÃO",
                "evidencia": "Parâmetro não encontrado na base."
            })
            continue

        # Agrupamento por ano
        for ano, grupo in dados.groupby("ano"):

            total = len(grupo)
            preenchidos = grupo[parametro].notna().sum()
            ausentes = grupo[parametro].isna().sum()

            if total > 0:
                cobertura = (preenchidos / total) * 100
            else:
                cobertura = 0

            # Classificação da ausência
            if preenchidos == 0:
                status = "AUSÊNCIA TOTAL"
            else:
                status = "DADOS DISPONÍVEIS"

            resultados.append({
                "regra": "CQ-04",
                "parametro": parametro,
                "ano": int(ano),
                "registros": total,
                "preenchidos": preenchidos,
                "ausentes": ausentes,
                "cobertura_pct": round(cobertura, 2),
                "status": status,
                "evidencia": (
                    f"{ausentes} ausentes de {total} registros."
                )
            })

    return pd.DataFrame(resultados)

In [ ]:
parametros = [
    "pH in loco",
    "Turbidez",
    "Oxigênio dissolvido",
    "Temperatura da água",
    "Condutividade elétrica in loco",
    "Demanda Bioquímica de Oxigênio",
    "Fósforo total",
    "Nitrato",
    "Sólidos totais",
    "Nitrogênio amoniacal total",
    "Coliformes termotolerantes"
]

resultado_cq04 = verificar_ausencia_temporal(
    df,
    parametros
)

display(resultado_cq04.head(20))

,regra,parametro,ano,registros,preenchidos,ausentes,cobertura_pct,status,evidencia
0,CQ-04,pH in loco,1997,421,421,0,100.00,DADOS DISPONÍVEIS,0 ausentes de 421 registros.
1,CQ-04,pH in loco,1998,745,745,0,100.00,DADOS DISPONÍVEIS,0 ausentes de 745 registros.
2,CQ-04,pH in loco,1999,795,795,0,100.00,DADOS DISPONÍVEIS,0 ausentes de 795 registros.
3,CQ-04,pH in loco,2000,959,959,0,100.00,DADOS DISPONÍVEIS,0 ausentes de 959 registros.
4,CQ-04,pH in loco,2001,962,962,0,100.00,DADOS DISPONÍVEIS,0 ausentes de 962 registros.
5,CQ-04,pH in loco,2002,971,971,0,100.00,DADOS DISPONÍVEIS,0 ausentes de 971 registros.
6,CQ-04,pH in loco,2003,1159,1159,0,100.00,DADOS DISPONÍVEIS,0 ausentes de 1159 registros.
7,CQ-04,pH in loco,2004,1161,1161,0,100.00,DADOS DISPONÍVEIS,0 ausentes de 1161 registros.
8,CQ-04,pH in loco,2005,1156,1156,0,100.00,DADOS DISPONÍVEIS,0 ausentes de 1156 registros.
9,CQ-04,pH in loco,2006,1322,1322,0,100.00,DADOS DISPONÍVEIS,0 ausentes de 1322 registros.


In [ ]:
# Verificar ausências totais
ausencias_totais = resultado_cq04[
    resultado_cq04["status"] == "AUSÊNCIA TOTAL"
].copy()

display(ausencias_totais)

,regra,parametro,ano,registros,preenchidos,ausentes,cobertura_pct,status,evidencia
247,CQ-04,Coliformes termotolerantes,2014,2637,0,2637,0.0,AUSÊNCIA TOTAL,2637 ausentes de 2637 registros.
248,CQ-04,Coliformes termotolerantes,2015,2626,0,2626,0.0,AUSÊNCIA TOTAL,2626 ausentes de 2626 registros.
249,CQ-04,Coliformes termotolerantes,2016,2542,0,2542,0.0,AUSÊNCIA TOTAL,2542 ausentes de 2542 registros.
250,CQ-04,Coliformes termotolerantes,2017,1734,0,1734,0.0,AUSÊNCIA TOTAL,1734 ausentes de 1734 registros.
251,CQ-04,Coliformes termotolerantes,2018,2745,0,2745,0.0,AUSÊNCIA TOTAL,2745 ausentes de 2745 registros.
252,CQ-04,Coliformes termotolerantes,2019,2786,0,2786,0.0,AUSÊNCIA TOTAL,2786 ausentes de 2786 registros.


In [ ]:
# Criar cópia para testar o funcionamento do checker
df_teste_cq04 = df.copy()

In [ ]:
import numpy as np
# Apagar deliberadamente os dados de Nitrato em 2010
df_teste_cq04.loc[
    df_teste_cq04["Data de Amostragem"].dt.year == 2010,
    "Nitrato"
] = np.nan

In [ ]:
resultado_teste_cq04 = verificar_ausencia_temporal(
    df_teste_cq04,
    parametros
)

In [ ]:
display(
    resultado_teste_cq04[
        (
            (resultado_teste_cq04["parametro"] == "Nitrato") &
            (resultado_teste_cq04["ano"] == 2010)
        )
    ]
)

,regra,parametro,ano,registros,preenchidos,ausentes,cobertura_pct,status,evidencia
174,CQ-04,Nitrato,2010,2183,0,2183,0.0,AUSÊNCIA TOTAL,2183 ausentes de 2183 registros.


In [ ]:
# ============================================
# CQ-05 - CONSISTÊNCIA FÍSICA
# ============================================

def verificar_consistencia_fisica(df):

    resultados = []

    # ----------------------------------------
    # Regra 1 - pH
    # ----------------------------------------

    parametro = "pH in loco"

    if parametro in df.columns:

        invalidos = df[
            (df[parametro] < 0) |
            (df[parametro] > 14)
        ]

        quantidade = len(invalidos)

        if quantidade == 0:
            status = "OK"
            resultado = "Nenhum valor de pH fora do intervalo [0, 14]."
        else:
            status = "ATENÇÃO"
            resultado = (
                f"{quantidade} valores de pH fora do intervalo [0, 14]."
            )

        resultados.append({
            "regra": "CQ-05",
            "parametro": parametro,
            "criterio": "0 <= pH <= 14",
            "quantidade_invalidos": quantidade,
            "status": status,
            "resultado": resultado
        })

    # ----------------------------------------
    # Regra 2 - Turbidez
    # ----------------------------------------

    parametro = "Turbidez"

    if parametro in df.columns:

        invalidos = df[
            df[parametro] < 0
        ]

        quantidade = len(invalidos)

        if quantidade == 0:
            status = "OK"
            resultado = "Nenhum valor negativo de turbidez."
        else:
            status = "ATENÇÃO"
            resultado = (
                f"{quantidade} valores negativos de turbidez."
            )

        resultados.append({
            "regra": "CQ-05",
            "parametro": parametro,
            "criterio": "Turbidez >= 0",
            "quantidade_invalidos": quantidade,
            "status": status,
            "resultado": resultado
        })

    # ----------------------------------------
    # Regra 3 - Temperatura da água
    # ----------------------------------------

    parametro = "Temperatura da água"

    if parametro in df.columns:

        invalidos = df[
            df[parametro] < 0
        ]

        quantidade = len(invalidos)

        if quantidade == 0:
            status = "OK"
            resultado = (
                "Nenhum valor negativo de temperatura da água."
            )
        else:
            status = "ATENÇÃO"
            resultado = (
                f"{quantidade} valores negativos de temperatura da água."
            )

        resultados.append({
            "regra": "CQ-05",
            "parametro": parametro,
            "criterio": "Temperatura >= 0",
            "quantidade_invalidos": quantidade,
            "status": status,
            "resultado": resultado
        })

    return pd.DataFrame(resultados)

In [ ]:
# Executar na base original
resultado_cq05 = verificar_consistencia_fisica(df)

display(resultado_cq05)

,regra,parametro,criterio,quantidade_invalidos,status,resultado
0,CQ-05,pH in loco,0 <= pH <= 14,0,OK,"Nenhum valor de pH fora do intervalo [0, 14]."
1,CQ-05,Turbidez,Turbidez >= 0,0,OK,Nenhum valor negativo de turbidez.
2,CQ-05,Temperatura da água,Temperatura >= 0,0,OK,Nenhum valor negativo de temperatura da água.


In [ ]:
# Criar cópia para verificar funcionamento da função
df_teste_cq05 = df.copy()

In [ ]:
# Introduzir valores artificialmente inválidos

df_teste_cq05.loc[0, "pH in loco"] = 20
df_teste_cq05.loc[1, "Turbidez"] = -10
df_teste_cq05.loc[2, "Temperatura da água"] = -5

In [ ]:
# Teste
resultado_teste_cq05 = verificar_consistencia_fisica(
    df_teste_cq05
)

display(resultado_teste_cq05)

,regra,parametro,criterio,quantidade_invalidos,status,resultado
0,CQ-05,pH in loco,0 <= pH <= 14,1,ATENÇÃO,"1 valores de pH fora do intervalo [0, 14]."
1,CQ-05,Turbidez,Turbidez >= 0,1,ATENÇÃO,1 valores negativos de turbidez.
2,CQ-05,Temperatura da água,Temperatura >= 0,1,ATENÇÃO,1 valores negativos de temperatura da água.


In [ ]:
# ============================================
# CQ-06 - IDENTIFICAÇÃO DE VALORES EXTREMOS
# ============================================

def verificar_outliers_iqr(df, parametros):

    resultados = []

    for parametro in parametros:

        if parametro not in df.columns:
            resultados.append({
                "regra": "CQ-06",
                "parametro": parametro,
                "q1": None,
                "q3": None,
                "limite_inferior": None,
                "limite_superior": None,
                "total_validos": 0,
                "outliers": 0,
                "percentual_outliers": None,
                "status": "ATENÇÃO",
                "resultado": "Parâmetro não encontrado."
            })
            continue

        serie = df[parametro].dropna()

        if len(serie) == 0:
            resultados.append({
                "regra": "CQ-06",
                "parametro": parametro,
                "q1": None,
                "q3": None,
                "limite_inferior": None,
                "limite_superior": None,
                "total_validos": 0,
                "outliers": 0,
                "percentual_outliers": None,
                "status": "ATENÇÃO",
                "resultado": "Não existem valores válidos para análise."
            })
            continue

        # Quartis
        q1 = serie.quantile(0.25)
        q3 = serie.quantile(0.75)

        # Intervalo interquartil
        iqr = q3 - q1

        # Limites
        limite_inferior = q1 - 1.5 * iqr
        limite_superior = q3 + 1.5 * iqr

        # Identificação dos candidatos
        outliers = serie[
            (serie < limite_inferior) |
            (serie > limite_superior)
        ]

        total_validos = len(serie)
        quantidade_outliers = len(outliers)
        percentual = (
            quantidade_outliers / total_validos
        ) * 100

        if quantidade_outliers == 0:
            status = "OK"
        else:
            status = "ATENÇÃO"

        resultados.append({
            "regra": "CQ-06",
            "parametro": parametro,
            "q1": round(q1, 4),
            "q3": round(q3, 4),
            "limite_inferior": round(limite_inferior, 4),
            "limite_superior": round(limite_superior, 4),
            "total_validos": total_validos,
            "outliers": quantidade_outliers,
            "percentual_outliers": round(percentual, 2),
            "status": status,
            "resultado": (
                f"{quantidade_outliers} possíveis outliers "
                f"({percentual:.2f}% dos valores válidos)."
            )
        })

    return pd.DataFrame(resultados)

In [ ]:
# Executar na base original
resultado_cq06 = verificar_outliers_iqr(
    df,
    parametros
)

display(resultado_cq06)

,regra,parametro,q1,q3,limite_inferior,limite_superior,total_validos,outliers,percentual_outliers,status,resultado
0,CQ-06,pH in loco,6.50,7.30,5.300,8.500,39525,625,1.58,ATENÇÃO,625 possíveis outliers (1.58% dos valores váli...
1,CQ-06,Turbidez,8.11,50.50,-55.475,114.085,39556,5318,13.44,ATENÇÃO,5318 possíveis outliers (13.44% dos valores vá...
2,CQ-06,Oxigênio dissolvido,6.30,7.80,4.050,10.050,39567,4066,10.28,ATENÇÃO,4066 possíveis outliers (10.28% dos valores vá...
3,CQ-06,Temperatura da água,21.70,26.30,14.800,33.200,39566,281,0.71,ATENÇÃO,281 possíveis outliers (0.71% dos valores váli...
4,CQ-06,Condutividade elétrica in loco,38.70,134.00,-104.250,276.950,39461,4360,11.05,ATENÇÃO,4360 possíveis outliers (11.05% dos valores vá...
5,CQ-06,Demanda Bioquímica de Oxigênio,2.00,2.80,0.800,4.000,39305,7280,18.52,ATENÇÃO,7280 possíveis outliers (18.52% dos valores vá...
6,CQ-06,Fósforo total,0.02,0.12,-0.130,0.270,39288,4442,11.31,ATENÇÃO,4442 possíveis outliers (11.31% dos valores vá...
7,CQ-06,Nitrato,0.11,0.68,-0.745,1.535,39276,3189,8.12,ATENÇÃO,3189 possíveis outliers (8.12% dos valores vál...
8,CQ-06,Sólidos totais,57.00,180.00,-127.500,364.500,38968,3097,7.95,ATENÇÃO,3097 possíveis outliers (7.95% dos valores vál...
9,CQ-06,Nitrogênio amoniacal total,0.10,0.30,-0.200,0.600,38376,6327,16.49,ATENÇÃO,6327 possíveis outliers (16.49% dos valores vá...


In [ ]:
# Parâmetros com maior quantidade de candidatos
resultado_cq06.sort_values(
    "percentual_outliers",
    ascending=False
)[
    [
        "parametro",
        "total_validos",
        "outliers",
        "percentual_outliers",
        "limite_inferior",
        "limite_superior",
        "status"
    ]
]

,parametro,total_validos,outliers,percentual_outliers,limite_inferior,limite_superior,status
5,Demanda Bioquímica de Oxigênio,39305,7280,18.52,0.800,4.000,ATENÇÃO
9,Nitrogênio amoniacal total,38376,6327,16.49,-0.200,0.600,ATENÇÃO
10,Coliformes termotolerantes,22022,3219,14.62,-18950.000,32170.000,ATENÇÃO
1,Turbidez,39556,5318,13.44,-55.475,114.085,ATENÇÃO
6,Fósforo total,39288,4442,11.31,-0.130,0.270,ATENÇÃO
4,Condutividade elétrica in loco,39461,4360,11.05,-104.250,276.950,ATENÇÃO
2,Oxigênio dissolvido,39567,4066,10.28,4.050,10.050,ATENÇÃO
7,Nitrato,39276,3189,8.12,-0.745,1.535,ATENÇÃO
8,Sólidos totais,38968,3097,7.95,-127.500,364.500,ATENÇÃO
0,pH in loco,39525,625,1.58,5.300,8.500,ATENÇÃO


In [ ]:
# Criar uma cópia para testar o funcionamento da função
df_teste_cq06 = df.copy()

In [ ]:
# Inserir um valor artificialmente extremo de pH
df_teste_cq06.loc[0, "pH in loco"] = 1000

In [ ]:
# Teste
resultado_teste_cq06 = verificar_outliers_iqr(
    df_teste_cq06,
    parametros
)

display(
    resultado_teste_cq06[
        resultado_teste_cq06["parametro"] == "pH in loco"
    ]
)

,regra,parametro,q1,q3,limite_inferior,limite_superior,total_validos,outliers,percentual_outliers,status,resultado
0,CQ-06,pH in loco,6.5,7.3,5.3,8.5,39525,626,1.58,ATENÇÃO,626 possíveis outliers (1.58% dos valores váli...


In [ ]:
# ============================================
# CQ-07 - FREQUÊNCIA DE AMOSTRAGEM
# ============================================

def verificar_frequencia_amostragem(df):

    dados = df.copy()

    # Criar ano
    dados["ano"] = pd.to_datetime(
        dados["Data de Amostragem"],
        errors="coerce"
    ).dt.year

    # Frequência por estação e ano
    frequencia = (
        dados
        .groupby(["Estação", "ano"])
        .size()
        .reset_index(name="n_amostras")
    )

    # Classificação operacional
    def classificar_frequencia(n):
        if n <= 2:
            return "BAIXA"
        elif n >= 8:
            return "ALTA"
        else:
            return "ESPERADA"

    frequencia["classificacao"] = (
        frequencia["n_amostras"]
        .apply(classificar_frequencia)
    )

    return frequencia

In [ ]:
# Executar na base original
resultado_cq07 = verificar_frequencia_amostragem(df)

display(resultado_cq07.head(20))

,Estação,ano,n_amostras,classificacao
0,AV005,2003,4,ESPERADA
1,AV005,2004,4,ESPERADA
2,AV005,2006,1,BAIXA
3,AV005,2007,4,ESPERADA
4,AV005,2008,4,ESPERADA
5,AV005,2009,4,ESPERADA
6,AV005,2010,4,ESPERADA
7,AV005,2011,4,ESPERADA
8,AV005,2012,4,ESPERADA
9,AV005,2013,3,ESPERADA


In [ ]:
print("Total de combinações estação-ano:", len(resultado_cq07))

Total de combinações estação-ano: 9883


In [ ]:
display(
    resultado_cq07["classificacao"]
    .value_counts()
)

,count
classificacao,
ESPERADA,8682
BAIXA,831
ALTA,370


Importante: As faixas BAIXA (1–2), ESPERADA (3–7) e ALTA (≥8) são classificações operacionais para triagem da auditoria, e não critérios de adequação do programa de monitoramento.

In [ ]:
# Combinações com extamente 4 amostras
quantidade_exatamente_4 = (
    resultado_cq07["n_amostras"] == 4
).sum()

percentual_exatamente_4 = (
    quantidade_exatamente_4 /
    len(resultado_cq07)
) * 100

print(
    f"Combinações com exatamente 4 amostras: "
    f"{quantidade_exatamente_4}"
)

print(
    f"Percentual: "
    f"{percentual_exatamente_4:.2f}%"
)

Combinações com exatamente 4 amostras: 7879
Percentual: 79.72%


In [ ]:
# ============================================
# CQ-08 - PRESERVAÇÃO DOS SINAIS DE MEDIÇÃO
# ============================================

def verificar_sinais_medicao(df):

    resultados = []

    # Identificar colunas de sinal
    colunas_sinal = [
        coluna for coluna in df.columns
        if coluna.startswith("Sinal ")
    ]

    if len(colunas_sinal) == 0:
        return pd.DataFrame([{
            "regra": "CQ-08",
            "parametro": None,
            "coluna_sinal": None,
            "sinais_encontrados": None,
            "total_com_sinal": 0,
            "sinal_sem_valor": 0,
            "status": "ATENÇÃO",
            "resultado": "Nenhuma coluna de sinal foi identificada."
        }])

    for coluna_sinal in colunas_sinal:

        # Nome do parâmetro correspondente
        parametro = coluna_sinal.replace("Sinal ", "", 1)

        # Se não existir a coluna correspondente ao valor
        if parametro not in df.columns:
            resultados.append({
                "regra": "CQ-08",
                "parametro": parametro,
                "coluna_sinal": coluna_sinal,
                "sinais_encontrados": None,
                "total_com_sinal": 0,
                "sinal_sem_valor": None,
                "status": "ATENÇÃO",
                "resultado": (
                    "Coluna de sinal sem coluna de valor correspondente."
                )
            })
            continue

        serie_sinal = df[coluna_sinal]
        serie_valor = df[parametro]

        # Valores de sinal presentes
        sinais_validos = (
            serie_sinal
            .dropna()
            .astype(str)
            .str.strip()
        )

        # Remover strings vazias
        sinais_validos = sinais_validos[
            sinais_validos != ""
        ]

        # Frequência dos sinais
        distribuicao = sinais_validos.value_counts()

        # Quantidade de sinais preenchidos
        total_com_sinal = len(sinais_validos)

        # Sinal preenchido, mas valor ausente
        sinal_sem_valor = (
            serie_sinal.notna() &
            serie_sinal.astype(str).str.strip().ne("") &
            serie_valor.isna()
        ).sum()

        # Montar representação da distribuição
        if len(distribuicao) > 0:
            sinais_encontrados = "; ".join(
                [
                    f"{sinal}: {quantidade}"
                    for sinal, quantidade
                    in distribuicao.items()
                ]
            )
        else:
            sinais_encontrados = "Nenhum sinal preenchido"

        if sinal_sem_valor > 0:
            status = "ATENÇÃO"
            resultado = (
                f"{sinal_sem_valor} registros possuem sinal "
                f"sem valor correspondente."
            )
        else:
            status = "OK"
            resultado = (
                "Sinais preservados e associados às respectivas "
                "colunas de valor."
            )

        resultados.append({
            "regra": "CQ-08",
            "parametro": parametro,
            "coluna_sinal": coluna_sinal,
            "sinais_encontrados": sinais_encontrados,
            "total_com_sinal": total_com_sinal,
            "sinal_sem_valor": sinal_sem_valor,
            "status": status,
            "resultado": resultado
        })

    return pd.DataFrame(resultados)

In [ ]:
# Executar na base original
resultado_cq08 = verificar_sinais_medicao(df)

print(
    f"Colunas de sinal analisadas: "
    f"{len(resultado_cq08)}"
)

display(resultado_cq08.head(20))

Colunas de sinal analisadas: 96


,regra,parametro,coluna_sinal,sinais_encontrados,total_com_sinal,sinal_sem_valor,status,resultado
0,CQ-08,"2,4,6 Triclorofenol","Sinal 2,4,6 Triclorofenol",<: 143,143,0,OK,Sinais preservados e associados às respectivas...
1,CQ-08,Alcalinidade de bicarbonato,Sinal Alcalinidade de bicarbonato,<: 14,14,0,OK,Sinais preservados e associados às respectivas...
2,CQ-08,Alcalinidade total,Sinal Alcalinidade total,<: 14,14,0,OK,Sinais preservados e associados às respectivas...
3,CQ-08,Aldrin + Dieldrin,Sinal Aldrin + Dieldrin,<: 142,142,0,OK,Sinais preservados e associados às respectivas...
4,CQ-08,Alumínio dissolvido,Sinal Alumínio dissolvido,<: 9184,9184,0,OK,Sinais preservados e associados às respectivas...
5,CQ-08,Alumínio total,Sinal Alumínio total,<: 38,38,0,OK,Sinais preservados e associados às respectivas...
6,CQ-08,Arsênio Dissolvido,Sinal Arsênio Dissolvido,<: 536,536,0,OK,Sinais preservados e associados às respectivas...
7,CQ-08,Arsênio total,Sinal Arsênio total,<: 17044,17044,0,OK,Sinais preservados e associados às respectivas...
8,CQ-08,Atrazina,Sinal Atrazina,<: 146,146,0,OK,Sinais preservados e associados às respectivas...
9,CQ-08,Bário total,Sinal Bário total,<: 1679,1679,0,OK,Sinais preservados e associados às respectivas...


In [ ]:
# Consolidar os sinais encontrados

sinais_observados = []

for coluna in colunas_sinal:

    if coluna in df.columns:

        sinais = (
            df[coluna]
            .dropna()
            .astype(str)
            .str.strip()
        )

        sinais = sinais[sinais != ""]

        for sinal, quantidade in sinais.value_counts().items():
            sinais_observados.append({
                "coluna_sinal": coluna,
                "sinal": sinal,
                "quantidade": quantidade
            })

sinais_observados = pd.DataFrame(sinais_observados)

display(
    sinais_observados
    .sort_values("quantidade", ascending=False)
    .head(30)
)

,coluna_sinal,sinal,quantidade
12,Sinal Cádmio total,<,27900
31,Sinal Demanda Bioquímica de Oxigênio,<,26008
14,Sinal Chumbo total,<,25972
20,Sinal Cobre dissolvido,<,24074
44,Sinal Fenóis totais,<,23712
28,Sinal Cromo total,<,22950
55,Sinal Mercúrio total,<,22682
75,Sinal Substâncias tensoativas,<,20439
59,Sinal Níquel total,<,19693
77,Sinal Sulfeto,<,18579


In [ ]:
# Criar uma cópia para testar o funcionamento da função
df_teste_cq08 = df.copy()

In [ ]:
# Encontrar uma coluna de sinal com pelo menos um registro disponível
coluna_teste_sinal = colunas_sinal[0]

parametro_teste = coluna_teste_sinal.replace(
    "Sinal ", "", 1
)

print("Coluna de sinal:", coluna_teste_sinal)
print("Parâmetro:", parametro_teste)

Coluna de sinal: Sinal 2,4,6 Triclorofenol
Parâmetro: 2,4,6 Triclorofenol


In [ ]:
# Encontrar uma linha onde o valor esteja preenchido
linha_teste = df_teste_cq08[
    df_teste_cq08[parametro_teste].notna()
].index[0]

# Criar artificialmente um sinal sem valor correspondente
df_teste_cq08.loc[
    linha_teste,
    parametro_teste
] = np.nan

df_teste_cq08.loc[
    linha_teste,
    coluna_teste_sinal
] = "<"

print("Linha alterada para teste:", linha_teste)

Linha alterada para teste: 1848


In [ ]:
# Teste
resultado_teste_cq08 = verificar_sinais_medicao(
    df_teste_cq08
)

display(
    resultado_teste_cq08[
        resultado_teste_cq08["coluna_sinal"] == coluna_teste_sinal
    ]
)

,regra,parametro,coluna_sinal,sinais_encontrados,total_com_sinal,sinal_sem_valor,status,resultado
0,CQ-08,"2,4,6 Triclorofenol","Sinal 2,4,6 Triclorofenol",<: 143,143,1,ATENÇÃO,1 registros possuem sinal sem valor correspond...


In [ ]:
# ============================================
# CQ-09 - PRESERVAÇÃO DE DADOS AUSENTES
# ============================================

def verificar_preservacao_ausentes(df_original, df_auditoria):

    resultados = []

    # Verificar se as estruturas possuem as mesmas colunas
    colunas_originais = set(df_original.columns)
    colunas_auditoria = set(df_auditoria.columns)

    colunas_faltantes = colunas_originais - colunas_auditoria

    if len(colunas_faltantes) > 0:
        return pd.DataFrame([{
            "regra": "CQ-09",
            "status": "BLOQUEADO",
            "colunas_analisadas": 0,
            "colunas_com_alteracao": None,
            "resultado": (
                "A estrutura da base auditada não possui todas "
                "as colunas da base original."
            ),
            "evidencia": ", ".join(colunas_faltantes)
        }])

    # Considerar somente as colunas comuns
    colunas = [
        coluna for coluna in df_original.columns
        if coluna in df_auditoria.columns
    ]

    alteracoes = []

    for coluna in colunas:

        nulos_original = df_original[coluna].isna().sum()
        nulos_auditoria = df_auditoria[coluna].isna().sum()

        if nulos_original != nulos_auditoria:
            alteracoes.append({
                "coluna": coluna,
                "nulos_original": nulos_original,
                "nulos_auditoria": nulos_auditoria,
                "diferenca": nulos_auditoria - nulos_original
            })

    if len(alteracoes) == 0:
        status = "OK"
        resultado = (
            "A quantidade de valores ausentes foi preservada "
            "em todas as colunas."
        )
    else:
        status = "ATENÇÃO"
        resultado = (
            f"{len(alteracoes)} colunas apresentaram alteração "
            "na quantidade de valores ausentes."
        )

    resultados.append({
        "regra": "CQ-09",
        "status": status,
        "colunas_analisadas": len(colunas),
        "colunas_com_alteracao": len(alteracoes),
        "resultado": resultado,
        "evidencia": alteracoes
    })

    return pd.DataFrame(resultados)

In [ ]:
# Executar na base original
resultado_cq09 = verificar_preservacao_ausentes(
    df,
    df.copy()
)

display(resultado_cq09)

,regra,status,colunas_analisadas,colunas_com_alteracao,resultado,evidencia
0,CQ-09,OK,196,0,A quantidade de valores ausentes foi preservad...,[]


In [ ]:
# Criar cópia para testar funcionamento da função
df_teste_cq09 = df.copy()

In [ ]:
# Encontrar uma linha com valor ausente
linha_teste = df_teste_cq09[
    df_teste_cq09["Coliformes termotolerantes"].isna()
].index[0]

# Preencher artificialmente o valor ausente
df_teste_cq09.loc[
    linha_teste,
    "Coliformes termotolerantes"
] = 999

print("Linha modificada:", linha_teste)

Linha modificada: 24


In [ ]:
# Teste
resultado_teste_cq09 = verificar_preservacao_ausentes(
    df,
    df_teste_cq09
)

display(resultado_teste_cq09)

,regra,status,colunas_analisadas,colunas_com_alteracao,resultado,evidencia
0,CQ-09,ATENÇÃO,196,1,1 colunas apresentaram alteração na quantidade...,"[{'coluna': 'Coliformes termotolerantes', 'nul..."


In [ ]:
# ============================================
# CQ-10 - RASTREABILIDADE
# ============================================

from datetime import datetime


def registrar_execucao_quality_checker(
    resultados,
    nome_execucao="quality_checker"
):

    timestamp = datetime.now().isoformat(
        timespec="seconds"
    )

    registros = []

    for _, linha in resultados.iterrows():

        registros.append({
            "execucao": nome_execucao,
            "timestamp": timestamp,
            "regra": linha.get("regra"),
            "status": linha.get("status"),
            "resultado": linha.get("resultado"),
            "evidencia": (
                linha.get("evidencia")
                if "evidencia" in linha.index
                else None
            )
        })

    return pd.DataFrame(registros)

In [ ]:
# Testar com CQ-04
log_cq04 = registrar_execucao_quality_checker(
    resultado_cq04,
    nome_execucao="teste_CQ04"
)

display(log_cq04.head(20))

,execucao,timestamp,regra,status,resultado,evidencia
0,teste_CQ04,2026-09-17T18:54:01,CQ-04,DADOS DISPONÍVEIS,None,0 ausentes de 421 registros.
1,teste_CQ04,2026-09-17T18:54:01,CQ-04,DADOS DISPONÍVEIS,None,0 ausentes de 745 registros.
2,teste_CQ04,2026-09-17T18:54:01,CQ-04,DADOS DISPONÍVEIS,None,0 ausentes de 795 registros.
3,teste_CQ04,2026-09-17T18:54:01,CQ-04,DADOS DISPONÍVEIS,None,0 ausentes de 959 registros.
4,teste_CQ04,2026-09-17T18:54:01,CQ-04,DADOS DISPONÍVEIS,None,0 ausentes de 962 registros.
5,teste_CQ04,2026-09-17T18:54:01,CQ-04,DADOS DISPONÍVEIS,None,0 ausentes de 971 registros.
6,teste_CQ04,2026-09-17T18:54:01,CQ-04,DADOS DISPONÍVEIS,None,0 ausentes de 1159 registros.
7,teste_CQ04,2026-09-17T18:54:01,CQ-04,DADOS DISPONÍVEIS,None,0 ausentes de 1161 registros.
8,teste_CQ04,2026-09-17T18:54:01,CQ-04,DADOS DISPONÍVEIS,None,0 ausentes de 1156 registros.
9,teste_CQ04,2026-09-17T18:54:01,CQ-04,DADOS DISPONÍVEIS,None,0 ausentes de 1322 registros.


In [ ]:
# Criar resumo geral
def resumir_resultado(regra, resultado, coluna_status="status"):

    if resultado is None or len(resultado) == 0:
        return {
            "regra": regra,
            "status": "NÃO EXECUTADO",
            "resultado": "Nenhum resultado produzido.",
            "evidencia": None
        }

    contagem_status = (
        resultado[coluna_status]
        .value_counts()
        .to_dict()
    )

    # Determinar status geral
    if "BLOQUEADO" in contagem_status:
        status_geral = "BLOQUEADO"
    elif "ATENÇÃO" in contagem_status:
        status_geral = "ATENÇÃO"
    elif "AUSÊNCIA TOTAL" in contagem_status:
        status_geral = "ATENÇÃO"
    else:
        status_geral = "OK"

    return {
        "regra": regra,
        "status": status_geral,
        "resultado": str(contagem_status),
        "evidencia": (
            f"{len(resultado)} registros de resultado produzidos."
        )
    }

In [ ]:
resumo_quality_checker = pd.DataFrame([

    resumir_resultado(
        "CQ-04",
        resultado_cq04
    ),

    resumir_resultado(
        "CQ-05",
        resultado_cq05
    ),

    resumir_resultado(
        "CQ-06",
        resultado_cq06
    ),

    {
        "regra": "CQ-07",
        "status": "OK",
        "resultado": (
            f"{len(resultado_cq07)} combinações estação-ano analisadas."
        ),
        "evidencia": (
            "Distribuição das classificações de frequência "
            "por estação e ano."
        )
    },

    {
        "regra": "CQ-08",
        "status": (
            "ATENÇÃO"
            if (resultado_cq08["status"] == "ATENÇÃO").any()
            else "OK"
        ),
        "resultado": (
            f"{len(resultado_cq08)} colunas de sinal analisadas."
        ),
        "evidencia": (
            "Distribuição dos sinais e verificação de "
            "sinais sem valor correspondente."
        )
    },

    {
        "regra": "CQ-09",
        "status": resultado_cq09.iloc[0]["status"],
        "resultado": resultado_cq09.iloc[0]["resultado"],
        "evidencia": str(
            resultado_cq09.iloc[0]["evidencia"]
        )
    }
])

display(resumo_quality_checker)

,regra,status,resultado,evidencia
0,CQ-04,ATENÇÃO,"{'DADOS DISPONÍVEIS': 247, 'AUSÊNCIA TOTAL': 6}",253 registros de resultado produzidos.
1,CQ-05,OK,{'OK': 3},3 registros de resultado produzidos.
2,CQ-06,ATENÇÃO,{'ATENÇÃO': 11},11 registros de resultado produzidos.
3,CQ-07,OK,9883 combinações estação-ano analisadas.,Distribuição das classificações de frequência ...
4,CQ-08,OK,96 colunas de sinal analisadas.,Distribuição dos sinais e verificação de sinai...
5,CQ-09,OK,A quantidade de valores ausentes foi preservad...,[]


In [ ]:
# Incorporar CQ-01, CQ-02 e CQ-03
resumo_inicial = pd.DataFrame([
    {
        "regra": "CQ-01",
        "status": resultado_quality_checker[
            resultado_quality_checker["regra"] == "CQ-01"
        ].iloc[0]["status"],
        "resultado": resultado_quality_checker[
            resultado_quality_checker["regra"] == "CQ-01"
        ].iloc[0]["resultado"],
        "evidencia": resultado_quality_checker[
            resultado_quality_checker["regra"] == "CQ-01"
        ].iloc[0]["evidencia"]
    },

    {
        "regra": "CQ-02",
        "status": resultado_quality_checker[
            resultado_quality_checker["regra"] == "CQ-02"
        ].iloc[0]["status"],
        "resultado": resultado_quality_checker[
            resultado_quality_checker["regra"] == "CQ-02"
        ].iloc[0]["resultado"],
        "evidencia": resultado_quality_checker[
            resultado_quality_checker["regra"] == "CQ-02"
        ].iloc[0]["evidencia"]
    },

    {
        "regra": "CQ-03",
        "status": "OK",
        "resultado": (
            "Cobertura calculada para os parâmetros selecionados."
        ),
        "evidencia": (
            "Resultado detalhado disponível em resultado_quality_checker."
        )
    }
])

In [ ]:
# Log final
log_final = pd.concat(
    [
        resumo_inicial,
        resumo_quality_checker
    ],
    ignore_index=True
)

log_final["timestamp"] = datetime.now().isoformat(
    timespec="seconds"
)

display(log_final)

,regra,status,resultado,evidencia,timestamp
0,CQ-01,OK,Todas as colunas obrigatórias estão presentes.,"Estação, Data de Amostragem, Hora de Amostragem",2026-09-17T19:00:47
1,CQ-02,OK,Nenhuma duplicidade identificada.,0 registros duplicados pela chave estação-data...,2026-09-17T19:00:47
2,CQ-03,OK,Cobertura calculada para os parâmetros selecio...,Resultado detalhado disponível em resultado_qu...,2026-09-17T19:00:47
3,CQ-04,ATENÇÃO,"{'DADOS DISPONÍVEIS': 247, 'AUSÊNCIA TOTAL': 6}",253 registros de resultado produzidos.,2026-09-17T19:00:47
4,CQ-05,OK,{'OK': 3},3 registros de resultado produzidos.,2026-09-17T19:00:47
5,CQ-06,ATENÇÃO,{'ATENÇÃO': 11},11 registros de resultado produzidos.,2026-09-17T19:00:47
6,CQ-07,OK,9883 combinações estação-ano analisadas.,Distribuição das classificações de frequência ...,2026-09-17T19:00:47
7,CQ-08,OK,96 colunas de sinal analisadas.,Distribuição dos sinais e verificação de sinai...,2026-09-17T19:00:47
8,CQ-09,OK,A quantidade de valores ausentes foi preservad...,[],2026-09-17T19:00:47


In [ ]:
# Salvar log como csv
log_final.to_csv(
    "quality_checker_log.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Log salvo como quality_checker_log.csv")

Log salvo como quality_checker_log.csv


In [ ]:
# ============================================
# PRÉ-VALIDAÇÃO DA ESTRUTURA WIDE
# ============================================

# Colunas identificadoras da observação
colunas_id = [
    "Estação",
    "Data de Amostragem",
    "Hora de Amostragem"
]

# Identifica as colunas de parâmetros (excluindo identificadores,
# coluna de ano e colunas de sinal)
colunas_parametros = [
    c for c in df.columns
    if c not in colunas_id
    and c != "ano"
    and not c.startswith("Sinal ")
]

# Para cada parâmetro, verifica se existe sua coluna de sinal
pares_parametros = []

for parametro in colunas_parametros:
    coluna_sinal = f"Sinal {parametro}"

    pares_parametros.append({
        "parametro": parametro,
        "coluna_valor": parametro,
        "coluna_sinal": coluna_sinal,
        "sinal_existe": coluna_sinal in df.columns
    })

pares_parametros = pd.DataFrame(pares_parametros)

# Resumo da estrutura
print(f"Total de colunas de parâmetros identificadas: {len(pares_parametros)}")
print(
    f"Parâmetros com coluna de sinal correspondente: "
    f"{pares_parametros['sinal_existe'].sum()}"
)
print(
    f"Parâmetros sem coluna de sinal correspondente: "
    f"{(~pares_parametros['sinal_existe']).sum()}"
)

print("\nColunas de parâmetros sem sinal correspondente:")
display(
    pares_parametros.loc[
        ~pares_parametros["sinal_existe"],
        ["parametro", "coluna_sinal"]
    ]
)

print("\nPrimeiros pares identificados:")
display(pares_parametros.head(15))

Total de colunas de parâmetros identificadas: 96
Parâmetros com coluna de sinal correspondente: 96
Parâmetros sem coluna de sinal correspondente: 0

Colunas de parâmetros sem sinal correspondente:


,parametro,coluna_sinal



Primeiros pares identificados:


,parametro,coluna_valor,coluna_sinal,sinal_existe
0,"2,4,6 Triclorofenol","2,4,6 Triclorofenol","Sinal 2,4,6 Triclorofenol",True
1,Alcalinidade de bicarbonato,Alcalinidade de bicarbonato,Sinal Alcalinidade de bicarbonato,True
2,Alcalinidade total,Alcalinidade total,Sinal Alcalinidade total,True
3,Aldrin + Dieldrin,Aldrin + Dieldrin,Sinal Aldrin + Dieldrin,True
4,Alumínio dissolvido,Alumínio dissolvido,Sinal Alumínio dissolvido,True
5,Alumínio total,Alumínio total,Sinal Alumínio total,True
6,Arsênio Dissolvido,Arsênio Dissolvido,Sinal Arsênio Dissolvido,True
7,Arsênio total,Arsênio total,Sinal Arsênio total,True
8,Atrazina,Atrazina,Sinal Atrazina,True
9,Bário total,Bário total,Sinal Bário total,True


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
# Na base original (wide), cada parâmetro ocupa uma coluna. Então, criaremos uma
# base transformada (long), na qual cada parâmetro vira uma linha. Isso facilita
# as análises posteriores, pois nos permite agrupar, comparar, calcular estatísticas
# e criar gráficos por parâmetro, estação ou período usando a mesma estrutura.
# ============================================
# TRANSFORMAÇÃO WIDE → LONG
# ============================================

registros_long = []

for _, linha in df.iterrows():

    # Dados identificadores da observação
    dados_base = {
        "Estação": linha["Estação"],
        "Data de Amostragem": linha["Data de Amostragem"],
        "Hora de Amostragem": linha["Hora de Amostragem"],
    }

    for parametro in colunas_parametros:

        coluna_sinal = f"Sinal {parametro}"

        registros_long.append({
            **dados_base,
            "parametro": parametro,
            "valor": linha[parametro],
            "sinal": linha[coluna_sinal]
        })

df_long = pd.DataFrame(registros_long)

print("Dimensões da base original:", df.shape)
print("Dimensões da base long:", df_long.shape)

print("\nColunas da base long:")
print(df_long.columns.tolist())

print("\nPrimeiras observações:")
display(df_long.head(10))

Dimensões da base original: (39615, 196)
Dimensões da base long: (3803040, 6)

Colunas da base long:
['Estação', 'Data de Amostragem', 'Hora de Amostragem', 'parametro', 'valor', 'sinal']

Primeiras observações:


,Estação,Data de Amostragem,Hora de Amostragem,parametro,valor,sinal
0,AV005,2003-01-13,11:20:00,"2,4,6 Triclorofenol",NaN,NaN
1,AV005,2003-01-13,11:20:00,Alcalinidade de bicarbonato,NaN,NaN
2,AV005,2003-01-13,11:20:00,Alcalinidade total,8.1,NaN
3,AV005,2003-01-13,11:20:00,Aldrin + Dieldrin,NaN,NaN
4,AV005,2003-01-13,11:20:00,Alumínio dissolvido,NaN,NaN
5,AV005,2003-01-13,11:20:00,Alumínio total,NaN,NaN
6,AV005,2003-01-13,11:20:00,Arsênio Dissolvido,NaN,NaN
7,AV005,2003-01-13,11:20:00,Arsênio total,0.0015,NaN
8,AV005,2003-01-13,11:20:00,Atrazina,NaN,NaN
9,AV005,2003-01-13,11:20:00,Bário total,0.01,NaN


In [ ]:
# ============================================
# VALIDAÇÃO DA BASE LONG
# ============================================

print("Linhas esperadas:", len(df) * len(colunas_parametros))
print("Linhas obtidas:", len(df_long))

print("\nValores ausentes:")
print(df_long[["valor", "sinal"]].isna().sum())

print("\nNúmero de parâmetros:")
print(df_long["parametro"].nunique())

print("\nNúmero de estações:")
print(df_long["Estação"].nunique())

print("\nPeríodo:")
print(
    df_long["Data de Amostragem"].min(),
    "até",
    df_long["Data de Amostragem"].max()
)

Linhas esperadas: 3803040
Linhas obtidas: 3803040

Valores ausentes:
valor    2273689
sinal    3385131
dtype: int64

Número de parâmetros:
96

Número de estações:
769

Período:
1997-07-02 00:00:00 até 2019-12-12 00:00:00


In [127]:
# ============================================
# ANÁLISE 1 — COMPLETUDE POR PARÂMETRO
# ============================================

completude = (
    df_long
    .groupby("parametro")["valor"]
    .agg(
        total_observacoes="size",
        valores_preenchidos="count"
    )
    .reset_index()
)

completude["valores_ausentes"] = (
    completude["total_observacoes"]
    - completude["valores_preenchidos"]
)

completude["cobertura_%"] = (
    completude["valores_preenchidos"]
    / completude["total_observacoes"]
    * 100
)

# Ordena da menor para a maior cobertura
completude = completude.sort_values("cobertura_%")

display(completude)

print("Parâmetros com menor cobertura:")
display(completude.head(10))

print("\nParâmetros com maior cobertura:")
display(completude.tail(10))

,parametro,total_observacoes,valores_preenchidos,valores_ausentes,cobertura_%
72,Precipitação,39615,0,39615,0.000000
88,Transparência,39615,11,39604,0.027767
91,Vanádio total,39615,14,39601,0.035340
55,Manganês dissolvido,39615,28,39587,0.070680
49,Fluoreto ionizado,39615,88,39527,0.222138
68,Pentaclorofenol,39615,91,39524,0.229711
60,Molinato,39615,133,39482,0.335731
3,Aldrin + Dieldrin,39615,142,39473,0.358450
0,"2,4,6 Triclorofenol",39615,143,39472,0.360974
40,Endrin,39615,145,39470,0.366023


Parâmetros com menor cobertura:


,parametro,total_observacoes,valores_preenchidos,valores_ausentes,cobertura_%
72,Precipitação,39615,0,39615,0.000000
88,Transparência,39615,11,39604,0.027767
91,Vanádio total,39615,14,39601,0.035340
55,Manganês dissolvido,39615,28,39587,0.070680
49,Fluoreto ionizado,39615,88,39527,0.222138
68,Pentaclorofenol,39615,91,39524,0.229711
60,Molinato,39615,133,39482,0.335731
3,Aldrin + Dieldrin,39615,142,39473,0.358450
0,"2,4,6 Triclorofenol",39615,143,39472,0.360974
40,Endrin,39615,145,39470,0.366023



Parâmetros com maior cobertura:


,parametro,total_observacoes,valores_preenchidos,valores_ausentes,cobertura_%
50,Fósforo total,39615,39288,327,99.174555
32,Demanda Bioquímica de Oxigênio,39615,39305,310,99.217468
87,Temperatura do ar,39615,39312,303,99.235138
22,Condição de tempo,39615,39330,285,99.280576
23,Condutividade elétrica in loco,39615,39461,154,99.611258
93,pH in loco,39615,39525,90,99.772813
16,Cloreto total,39615,39536,79,99.800581
90,Turbidez,39615,39556,59,99.851067
86,Temperatura da água,39615,39566,49,99.876309
67,Oxigênio dissolvido,39615,39567,48,99.878834


In [128]:
# ============================================
# ANÁLISE 2 — COBERTURA AO LONGO DO TEMPO
# ============================================

parametros_principais = [
    "pH in loco",
    "Turbidez",
    "Oxigênio dissolvido",
    "Temperatura da água",
    "Condutividade elétrica in loco",
    "Demanda Bioquímica de Oxigênio",
    "Fósforo total",
    "Nitrato",
    "Sólidos totais",
    "Nitrogênio amoniacal total",
    "Coliformes termotolerantes"
]

df_long["ano"] = df_long["Data de Amostragem"].dt.year

cobertura_temporal = (
    df_long[df_long["parametro"].isin(parametros_principais)]
    .groupby(["ano", "parametro"])["valor"]
    .agg(
        total_observacoes="size",
        valores_preenchidos="count"
    )
    .reset_index()
)

cobertura_temporal["cobertura_%"] = (
    cobertura_temporal["valores_preenchidos"]
    / cobertura_temporal["total_observacoes"]
    * 100
)

display(cobertura_temporal.head(20))

print("Menores coberturas anuais:")

display(
    cobertura_temporal
    .sort_values("cobertura_%")
    .head(20)
)

,ano,parametro,total_observacoes,valores_preenchidos,cobertura_%
0,1997,Coliformes termotolerantes,421,421,100.000000
1,1997,Condutividade elétrica in loco,421,376,89.311164
2,1997,Demanda Bioquímica de Oxigênio,421,416,98.812352
3,1997,Fósforo total,421,404,95.961995
4,1997,Nitrato,421,418,99.287411
5,1997,Nitrogênio amoniacal total,421,421,100.000000
6,1997,Oxigênio dissolvido,421,421,100.000000
7,1997,Sólidos totais,421,321,76.247031
8,1997,Temperatura da água,421,421,100.000000
9,1997,Turbidez,421,421,100.000000


Menores coberturas anuais:


,ano,parametro,total_observacoes,valores_preenchidos,cobertura_%
242,2019,Coliformes termotolerantes,2786,0,0.000000
209,2016,Coliformes termotolerantes,2542,0,0.000000
198,2015,Coliformes termotolerantes,2626,0,0.000000
231,2018,Coliformes termotolerantes,2745,0,0.000000
220,2017,Coliformes termotolerantes,1734,0,0.000000
187,2014,Coliformes termotolerantes,2637,0,0.000000
176,2013,Coliformes termotolerantes,2586,614,23.743233
7,1997,Sólidos totais,421,321,76.247031
110,2007,Coliformes termotolerantes,1534,1298,84.615385
137,2009,Nitrogênio amoniacal total,2055,1819,88.515815


Achado: ausência total de valores de Coliformes termotolerantes entre 2014 e 2019 na base analisada.

Evidência: cobertura anual igual a 0% no período.

Interpretação: indica uma lacuna temporal relevante que deve ser investigada.

Limitação: a análise não permite determinar a causa da ausência.

In [129]:
# ============================================
# VALIDAÇÃO DOS ACHADOS
# ============================================

achados_validados = pd.DataFrame([
    {
        "id": "AV-01",
        "achado": "Baixa cobertura de alguns parâmetros",
        "evidencia": "Há parâmetros com cobertura muito baixa ou nula na base.",
        "interpretacao": "A disponibilidade dos parâmetros é heterogênea.",
        "conclusao_permitida": "A cobertura deve ser considerada antes de realizar análises específicas.",
        "conclusao_nao_permitida": "Concluir que os parâmetros com baixa cobertura são inválidos."
    },
    {
        "id": "AV-02",
        "achado": "Lacuna temporal de Coliformes termotolerantes",
        "evidencia": "A cobertura anual é 0% para Coliformes termotolerantes entre 2014 e 2019.",
        "interpretacao": "Existe uma lacuna temporal concentrada na disponibilidade desse parâmetro.",
        "conclusao_permitida": "O período deve ser sinalizado para investigação antes de análises que dependam desse parâmetro.",
        "conclusao_nao_permitida": "Afirmar a causa da ausência dos dados."
    },
    {
        "id": "AV-03",
        "achado": "Existência de valores extremos",
        "evidencia": "A auditoria identificou observações classificadas como candidatas a valores extremos pelo critério de IQR.",
        "interpretacao": "Algumas observações apresentam comportamento estatisticamente incomum.",
        "conclusao_permitida": "Os valores devem ser investigados no contexto da estação, período e parâmetro.",
        "conclusao_nao_permitida": "Classificar automaticamente um valor extremo como erro de medição."
    }
])

display(achados_validados)

,id,achado,evidencia,interpretacao,conclusao_permitida,conclusao_nao_permitida
0,AV-01,Baixa cobertura de alguns parâmetros,Há parâmetros com cobertura muito baixa ou nul...,A disponibilidade dos parâmetros é heterogênea.,A cobertura deve ser considerada antes de real...,Concluir que os parâmetros com baixa cobertura...
1,AV-02,Lacuna temporal de Coliformes termotolerantes,A cobertura anual é 0% para Coliformes termoto...,Existe uma lacuna temporal concentrada na disp...,O período deve ser sinalizado para investigaçã...,Afirmar a causa da ausência dos dados.
2,AV-03,Existência de valores extremos,A auditoria identificou observações classifica...,Algumas observações apresentam comportamento e...,Os valores devem ser investigados no contexto ...,Classificar automaticamente um valor extremo c...


In [130]:
# ============================================
# VALIDADOR DE ACHADOS
# ============================================

campos_obrigatorios = [
    "id",
    "achado",
    "evidencia",
    "interpretacao",
    "conclusao_permitida",
    "conclusao_nao_permitida"
]

validacao = []

for _, linha in achados_validados.iterrows():

    campos_preenchidos = all(
        pd.notna(linha[c]) and str(linha[c]).strip() != ""
        for c in campos_obrigatorios
    )

    validacao.append({
        "id": linha["id"],
        "status": "OK" if campos_preenchidos else "BLOQUEADO"
    })

resultado_validacao = pd.DataFrame(validacao)

display(resultado_validacao)

print(
    f"\nAchados validados: "
    f"{(resultado_validacao['status'] == 'OK').sum()}"
    f"/{len(resultado_validacao)}"
)

,id,status
0,AV-01,OK
1,AV-02,OK
2,AV-03,OK



Achados validados: 3/3


In [131]:
# ============================================
# HARNESS / ORQUESTRAÇÃO
# ============================================
# Criar o controlador do sistema
# ============================================

def executar_pipeline():
    """
    Orquestra o fluxo de análise da qualidade da água.
    As operações analíticas são determinísticas;
    a interpretação por LLM pode ser adicionada posteriormente.
    """

    resultados = []

    # ----------------------------------------
    # ETAPA 1 — VERIFICAÇÃO DA ESTRUTURA
    # ----------------------------------------
    estrutura_ok = (
        len(pares_parametros) > 0
        and pares_parametros["sinal_existe"].all()
    )

    resultados.append({
        "etapa": "estrutura",
        "status": "OK" if estrutura_ok else "BLOQUEADO",
        "evidencia": (
            f"{len(pares_parametros)} parâmetros identificados; "
            f"{pares_parametros['sinal_existe'].sum()} com sinal correspondente."
        )
    })

    if not estrutura_ok:
        return pd.DataFrame(resultados)

    # ----------------------------------------
    # ETAPA 2 — QUALITY CHECKER
    # ----------------------------------------
    quality_ok = len(regras_qualidade) == 9

    resultados.append({
        "etapa": "quality_checker",
        "status": "OK" if quality_ok else "BLOQUEADO",
        "evidencia": f"{len(regras_qualidade)} regras de qualidade definidas."
    })

    if not quality_ok:
        return pd.DataFrame(resultados)

    # ----------------------------------------
    # ETAPA 3 — ANÁLISE
    # ----------------------------------------
    analise_ok = (
        len(completude) > 0
        and len(cobertura_temporal) > 0
    )

    resultados.append({
        "etapa": "analise",
        "status": "OK" if analise_ok else "BLOQUEADO",
        "evidencia": (
            f"{len(completude)} parâmetros avaliados quanto à completude; "
            f"{len(cobertura_temporal)} registros de cobertura temporal."
        )
    })

    if not analise_ok:
        return pd.DataFrame(resultados)

    # ----------------------------------------
    # ETAPA 4 — VALIDAÇÃO DOS ACHADOS
    # ----------------------------------------
    validacao_ok = (
        len(resultado_validacao) > 0
        and (resultado_validacao["status"] == "OK").all()
    )

    resultados.append({
        "etapa": "validacao_dos_achados",
        "status": "OK" if validacao_ok else "BLOQUEADO",
        "evidencia": (
            f"{(resultado_validacao['status'] == 'OK').sum()} "
            f"de {len(resultado_validacao)} achados validados."
        )
    })

    if not validacao_ok:
        return pd.DataFrame(resultados)

    # ----------------------------------------
    # ETAPA 5 — PRONTO PARA RELATÓRIO
    # ----------------------------------------
    resultados.append({
        "etapa": "relatorio",
        "status": "PRONTO",
        "evidencia": "Todas as etapas anteriores foram concluídas."
    })

    return pd.DataFrame(resultados)


pipeline = executar_pipeline()

display(pipeline)

,etapa,status,evidencia
0,estrutura,OK,96 parâmetros identificados; 96 com sinal corr...
1,quality_checker,OK,9 regras de qualidade definidas.
2,analise,OK,96 parâmetros avaliados quanto à completude; 2...
3,validacao_dos_achados,OK,3 de 3 achados validados.
4,relatorio,PRONTO,Todas as etapas anteriores foram concluídas.


In [132]:
# ============================================
# CRITÉRIO DE CONCLUSÃO DO PIPELINE
# ============================================

pipeline_concluido = (
    (pipeline["status"].isin(["OK", "PRONTO"])).all()
    and pipeline.iloc[-1]["status"] == "PRONTO"
)

print(
    "PIPELINE CONCLUÍDO"
    if pipeline_concluido
    else "PIPELINE NÃO CONCLUÍDO"
)

PIPELINE CONCLUÍDO


In [133]:
# ============================================
# RELATÓRIO FINAL
# ============================================

relatorio_final = f"""
# Relatório de Análise Exploratória e Controle de Qualidade
## Monitoramento da Qualidade das Águas em Minas Gerais

### 1. Objetivo

Este relatório apresenta os resultados produzidos pelo sistema desenvolvido
para análise exploratória e controle de qualidade de dados de monitoramento
da qualidade das águas.

O sistema combina verificações determinísticas realizadas por código,
análise exploratória e validação dos achados antes da elaboração das
conclusões.

---

### 2. Base analisada

A demonstração foi realizada a partir da base histórica do Programa
Águas de Minas, do IGAM, utilizando a planilha "SH ATÉ DEZ 2019".

Características da base:

- Registros de amostragem: {len(df):,}
- Estações: {df["Estação"].nunique()}
- Parâmetros disponíveis: {len(colunas_parametros)}
- Período: {df["Data de Amostragem"].min().strftime("%d/%m/%Y")}
  a {df["Data de Amostragem"].max().strftime("%d/%m/%Y")}

Para a análise exploratória detalhada foram utilizados
{len(parametros_principais)} parâmetros principais.

---

### 3. Controle de qualidade

Foram executadas {len(regras_qualidade)} regras de qualidade,
abrangendo aspectos de:

- estrutura;
- completude;
- completude temporal;
- consistência;
- valores extremos;
- cobertura amostral;
- semântica;
- dados ausentes.

A estrutura principal apresentou {len(pares_parametros)} parâmetros
com suas respectivas colunas de sinal.

Não foram identificadas duplicidades pela chave
Estação + Data de Amostragem + Hora de Amostragem.

Os valores ausentes foram preservados durante o processo de
padronização dos dados.

---

### 4. Padronização dos dados

A estrutura original, em formato wide, foi transformada para o formato
long, resultando em {len(df_long):,} registros analíticos.

A transformação preservou:

- estação;
- data e hora da amostragem;
- parâmetro;
- valor observado;
- sinal associado à observação;
- valores ausentes.

Essa estrutura permite realizar análises por parâmetro, estação e período
de maneira padronizada.

---

### 5. Principais achados

#### 5.1 Cobertura dos parâmetros

A disponibilidade dos parâmetros é heterogênea.

Na base analisada, alguns parâmetros apresentam cobertura muito baixa,
enquanto parâmetros como Oxigênio dissolvido, Temperatura da água e
Turbidez apresentam cobertura superior a 99%.

Essa diferença deve ser considerada antes da realização de análises
específicas.

Baixa cobertura não é interpretada automaticamente como erro ou
invalidação do parâmetro.

#### 5.2 Lacuna temporal

Foi identificada ausência total de valores de Coliformes termotolerantes
entre 2014 e 2019 na base analisada.

A cobertura anual do parâmetro é igual a 0% nesse período.

O resultado indica uma lacuna temporal relevante na disponibilidade
do parâmetro.

A análise realizada não permite determinar a causa dessa ausência.

#### 5.3 Valores extremos

A auditoria identificou observações classificadas como candidatas a
valores extremos pelo critério de intervalo interquartil (IQR).

Essas observações são tratadas como pontos que merecem investigação
contextual, considerando estação, período e parâmetro.

Um valor extremo não é automaticamente classificado como erro de
medição.

---

### 6. Validação dos achados

Foram avaliados {len(achados_validados)} achados estruturados.

Cada achado foi associado a:

- evidência observável;
- interpretação;
- conclusão permitida;
- conclusão que deve ser evitada.

Esse mecanismo busca impedir que uma observação estatística seja
transformada automaticamente em uma afirmação causal ou em um
diagnóstico ambiental.

---

### 7. Limitações

Os resultados devem ser interpretados considerando o escopo da base
e as limitações do processo.

O sistema não permite:

- estabelecer causalidade;
- determinar a causa de dados ausentes;
- classificar automaticamente valores extremos como erros;
- produzir diagnóstico ambiental definitivo;
- inferir riscos à saúde;
- afirmar conformidade regulatória;
- substituir avaliação técnica especializada.

---

### 8. Critério de conclusão

O pipeline considera a análise concluída somente após a execução das
etapas de:

**estrutura → controle de qualidade → análise → validação dos achados.**

Status final do pipeline: **{pipeline.iloc[-1]["status"]}**

A execução foi considerada concluída porque as etapas previstas
retornaram os critérios de aceite definidos pelo sistema.
"""

print(relatorio_final)


# Relatório de Análise Exploratória e Controle de Qualidade
## Monitoramento da Qualidade das Águas em Minas Gerais

### 1. Objetivo

Este relatório apresenta os resultados produzidos pelo sistema desenvolvido
para análise exploratória e controle de qualidade de dados de monitoramento
da qualidade das águas.

O sistema combina verificações determinísticas realizadas por código,
análise exploratória e validação dos achados antes da elaboração das
conclusões.

---

### 2. Base analisada

A demonstração foi realizada a partir da base histórica do Programa
Águas de Minas, do IGAM, utilizando a planilha "SH ATÉ DEZ 2019".

Características da base:

- Registros de amostragem: 39,615
- Estações: 769
- Parâmetros disponíveis: 96
- Período: 02/07/1997
  a 12/12/2019

Para a análise exploratória detalhada foram utilizados
11 parâmetros principais.

---

### 3. Controle de qualidade

Foram executadas 9 regras de qualidade,
abrangendo aspectos de:

- estrutura;
- completude;
- completude temporal

In [134]:
# Salvar relatório final em Markdown

with open("relatorio_final.md", "w", encoding="utf-8") as arquivo:
    arquivo.write(relatorio_final)

print("Relatório salvo como: relatorio_final.md")

Relatório salvo como: relatorio_final.md
